# [실습] 원점 5개 평가표 — 베이스라인과 첫 신경망

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

## 글로벌 MLP 를 인력 예측에 쓸까

<div style="background:#FFFFFF;color:#000000;padding:40px;border-radius:12px;border:1px solid #E6E6E6;border-left:4px solid #051C2C">
<div style="font-size:0.9em;color:#656565">딥러닝 시계열 · 첫날</div>
<h1 style="color:#000000;margin:10px 0">원점 5개 평가표 — 베이스라인과 첫 신경망</h1>
<div style="font-size:1.05em;color:#656565">서울 지하철 100개 역(2024년까지 하루 평균 승차가 가장 많은 역 100곳)을 모델 하나로 학습한 글로벌 MLP 를 손실만 바꿔 채점하고, 입력 윈도의 누수를 고칩니다</div>
</div>

**오늘의 질문** 100개 역을 한 모델로 학습한 신경망을 다음 주 역별 인력 예측에 쓸까요?
예측하는 값은 역별 하루 승차입니다. 역 운영팀이 지금 쓰는 방식은 바로 전 7일의 평균 승차를 다음 7일 모두에 적는 **직전 7일 평균**이며, 표 1 에도 이 이름으로 나옵니다. 신경망은 입력 28일로 다음 7일을 내는 글로벌 MLP 입니다.

**예측 원점(forecast origin)** 은 예측하는 7일의 첫날이고, 예측에는 원점 전날까지의 자료만 씁니다. 원점마다 채점하는 7일이 한 주입니다. 원점 5개는 **공휴일 없는 두 주**와, 채점 7일에 평일 공휴일이 있는 **공휴일 있는 세 주**로 나뉩니다. 두 묶음의 역 단위 MASE 평균을 같은 평가표의 계절 나이브와 견줘 답합니다.

1. 맨 위 설치 셀을 한 번 실행합니다. 런타임 유형은 CPU 로 둬도 됩니다.
2. 「오늘의 준비」 셀을 실행해 두고, 출력을 기다리는 동안 개념 확인 두 문항을 풉니다.
3. 노트북을 내려받아 Colab 이나 내 컴퓨터에서 열었다면 맨 위의 접힌 `[채점]` 셀을 한 번 ▶ 실행한 뒤 시작합니다.

In [ ]:
# 실습 준비 - 패키지·폰트·장치 확인
#  오늘은 Colab 기본 패키지(numpy · pandas · pyarrow · matplotlib · torch)로 충분해 따로 설치하지 않습니다
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.font_manager as fm
import torch
from torch import nn
import warnings; warnings.filterwarnings("ignore")

# 한글 폰트: Colab 이면 나눔고딕을 설치한 뒤 그 ttf 를 폰트 목록에 직접 등록한다.
#  설치는 디스크에 파일만 만든다. 이미 떠 있는 폰트 목록은 그 사실을 모르므로 addfont 로
#  넣어 줘야 이름이 보인다(넣지 않으면 DejaVu Sans 로 밀려 한글이 전부 □ 로 나온다).
import glob, shutil, subprocess
if shutil.which("apt-get"):
    subprocess.run(["apt-get", "-y", "-qq", "install", "fonts-nanum"], check=False)
for _p in glob.glob("/usr/share/fonts/truetype/nanum/Nanum*.ttf"):   # 설치 여부와 무관하게 한 번
    try: fm.fontManager.addfont(_p)
    except Exception: pass
try: fm._get_font.cache_clear()
except Exception: pass
installed = {f.name for f in fm.fontManager.ttflist}
FONT = next((n for n in ["Pretendard", "Apple SD Gothic Neo", "NanumGothic"] if n in installed), "DejaVu Sans")
plt.rcParams["font.family"] = FONT
plt.rcParams["axes.unicode_minus"] = FONT in {"Pretendard", "DejaVu Sans"}   # 나머지 폰트에는 U+2212 가 없다
%config InlineBackend.print_figure_kwargs = {"bbox_inches": None}   # 그림마다 폭이 깎이지 않게
plt.rcParams["figure.dpi"] = plt.rcParams["savefig.dpi"] = 150      # 폭 FIG_W x 150dpi = 1500px 고정
plt.rcParams["figure.autolayout"] = True
plt.rcParams.update({"font.size": 11, "axes.titlesize": 15, "axes.labelsize": 12,
                     "legend.fontsize": 11, "figure.titlesize": 17, "font.weight": "regular",
                     "text.color": "black", "axes.labelcolor": "black", "axes.titlecolor": "black",
                     "axes.edgecolor": "#757575", "xtick.color": "#757575", "ytick.color": "#757575",
                     "xtick.labelcolor": "black", "ytick.labelcolor": "black",
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": False})

# 오늘 그림 전체가 함께 쓰는 색: 기준 모델=남색, 관측=파랑, 강조=하늘색, 문맥=회색
COL = {"남색": "#051C2C", "파랑": "#2251FF", "하늘색": "#00A9F4",
       "회색": "#757575", "연회색": "#B3B3B3"}
FIG_W = 10.0     # 캔버스 폭은 오늘 그림 모두 이 값으로 고정하고 높이만 그림마다 조절한다
FIGW = FIG_W      # 도우미가 밑줄을 빼고 FIGW 로 옮겨 적는 일이 있어 같은 값을 하나 더 둔다

print("사용 폰트:", FONT, "| 색 팔레트 준비 완료")

# 장치 확인: 신경망 셀이 모델과 입력을 옮길 곳(DEVICE)을 정합니다
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cuda.matmul.allow_tf32 = False   # fp32 로 계산해 GPU 종류에 따라 값이 갈리지 않게
torch.backends.cudnn.allow_tf32 = False

if DEVICE == "cuda":
    _gpu = torch.cuda.get_device_properties(0)
    print(f"DEVICE : cuda | GPU {torch.cuda.get_device_name(0)} | 메모리 {_gpu.total_memory / 1e9:.1f} GB"
          f" | torch {torch.__version__}")
else:
    print(f"DEVICE : cpu | torch {torch.__version__} | 오늘의 작은 MLP 는 CPU 로 충분합니다")
    print("GPU 를 쓰려면 메뉴 런타임 → 런타임 유형 변경 → GPU 를 고른 뒤 맨 위 셀부터 다시 실행합니다")

In [ ]:
#@title [채점] 이 셀을 먼저 실행하세요 (한 번이면 됩니다)
#   미션의 「PROMPT 셀」을 실행하면 그 셀의 출력과 그림을 받아 실행 결과에 채점 결과를 찍습니다.
#   통과하면 그 자리에서 완성본 풀이가 열리고, 아니면 항목별 힌트와 튜터에 보낼 질문이 나옵니다.
#   읽지 않고 실행만 해도 됩니다 — 기대값과 풀이는 이 셀에만 있고 「PROMPT 셀」에는 없습니다.
import base64
import glob
import io
import json
import re
import shutil
import subprocess
import sys

import matplotlib.font_manager as fm      # 아래 폰트 자급이 쓴다
import matplotlib.pyplot as plt


def _font_bootstrap():
    """한글 폰트 자급 — 맨 위 설치·폰트 셀을 건너뛰어도 그래프의 한글이 깨지지 않게.

    LMS 는 이 셀만 자동 실행하므로 폰트 준비도 여기서 한 번 더 한다. 설치·폰트 셀과 같은
    규칙이라 둘 다 돌아도 결과가 같고, apt-get 이 없는 자리(로컬)에서는 곧바로 지나간다.
    설치한 ttf 는 `addfont` 로 폰트 목록에 직접 넣는다 — 넣지 않으면 목록이 옛 상태라
    Colab 에서도 한글이 □ 로 깨진다.
    """
    try:
        cands = ["Pretendard", "Apple SD Gothic Neo", "NanumGothic"]
        installed = {f.name for f in fm.fontManager.ttflist}
        if not any(n in installed for n in cands) and shutil.which("apt-get"):
            subprocess.run(["apt-get", "-y", "-qq", "install", "fonts-nanum"],
                           check=False, timeout=120)
        # 설치는 파일만 만든다 — 이미 떠 있는 폰트 목록에 addfont 로 넣어야 이름이 보인다.
        # (설치를 건너뛴 자리도 나눔 ttf 가 이미 있으면 이 줄들만으로 잡힌다.)
        for _p in glob.glob("/usr/share/fonts/truetype/nanum/Nanum*.ttf"):
            try: fm.fontManager.addfont(_p)
            except Exception: pass
        try: fm._get_font.cache_clear()
        except Exception: pass
        installed = {f.name for f in fm.fontManager.ttflist}
        _font = next((n for n in cands if n in installed), "DejaVu Sans")
        plt.rcParams["font.family"] = _font
        plt.rcParams["axes.unicode_minus"] = _font in {"Pretendard", "DejaVu Sans"}
    except Exception:
        print("한글 폰트를 못 찾아 그래프 글자가 깨질 수 있습니다 —"
              " 맨 위 설치·폰트 셀을 실행하세요")


_font_bootstrap()


def _tail_numbers(tails):
    """라벨 뒤 꼬리에서 괄호 설명을 걷어내고 첫 숫자 하나씩만 모읍니다.

    꼬리 전체의 숫자를 담으면 괄호 안 설명에 우연히 적힌 수가 기준 범위 안에 있을 때 틀린 값을
    통과시킵니다. 그래서 괄호 안은 버리고 맨 앞 숫자 하나만 봅니다.
    """
    picked = []
    for one in tails:
        plain, depth = [], 0
        for ch in one:
            if ch in "([":
                depth += 1
            elif ch in ")]":
                depth = max(0, depth - 1)
            elif depth == 0:
                plain.append(ch)
        got = _nums("".join(plain))
        if got:
            picked.append(got[0])
    return picked


def _dec_json(s):                          # base64(JSON) → 값. 리스트는 원래 튜플이라 되돌립니다
    def _t(v):
        return ({k: _t(x) for k, x in v.items()} if isinstance(v, dict)
                else tuple(_t(x) for x in v) if isinstance(v, list) else v)
    return _t(json.loads(base64.b64decode(s).decode("utf-8")))


# 채점 기준 — base64(JSON), 통과 판정에만 쓴다
_ANS = _dec_json(
               "eyJtMSI6eyJ2YWx1ZXMiOnsi6rO17Zy07J28IOyXhuuKlCDso7wgTUFTRSDssKgoTVNFIOKIkiBNQUUpIjpbMC4xLDAuMzVdLCLq"
               "s7XtnLTsnbwg7J6I64qUIOyjvCBNQVNFIOywqChNU0Ug4oiSIE1BRSkiOlstMC40NSwtMC4wOF19LCJwbG90Ijp7Iuy2leudvOuy"
               "qCI6dHJ1ZX19LCJtMiI6eyJ2YWx1ZXMiOnsi7JuQ7KCQIOyghOuCoOyXkOyEnCDrgZ3rgpjripQg7J6F66ClIOyciOuPhCDsiJgi"
               "Ols1LDUsIiJdLCLqs7XtnLTsnbwg7JeG64qUIOuRkCDso7wg7Y+J6regIE1BU0UiOlswLjcsMC44LCIiXX0sImZvcmJpZCI6WyLs"
               "i6Dqsr3rp53snYAg67KE66a964uI64ukIiwi7YGs6rKMIOu5l+uCmOqwkeuLiOuLpCJdLCJwbG90Ijp7Iuy2leudvOuyqCI6dHJ1"
               "ZX19fQ==")
_REVEAL = {                                # 완성본 풀이 — base64, 통과할 때만 디코드해 인쇄합니다
    "m1": ("8J+UkyDrr7jshZggMSDsmYTshLHrs7g6IEFJIOy9lOuTnCDspIQg7JWE656Y7JeQIOuTpOyWtOyZgOyVvCDtlaAg7L2U65Oc64qU"
           "IOydtCDsoJXrj4TsnoXri4jri6QKCuKWoCDsnbQg7L2U65Oc66W8IOu2gOuluCDtlITroaztlITtirgg7JiICgogICAg7KSA67mE"
           "IOyFgOydmCBmaXRfZ2xvYmFsLCBPUklHSU5fUExBSU4sIE9SSUdJTl9IT0xJREFZLCBTTkFJVkVfVFdPLCBTRUVELCBERVZJQ0Ug"
           "64qUIOydtOuvuCDsnojsnLzri4gg7IOI66GcIOunjOuTpOyngCDrp5DslYQg7KSYLgogICAg6rO17Zy07J28IOyXhuuKlCDso7zs"
           "nZgg7JuQ7KCQIE9SSUdJTl9QTEFJTiDqs7wg6rO17Zy07J28IOyeiOuKlCDso7zsnZgg7JuQ7KCQIE9SSUdJTl9IT0xJREFZIOun"
           "iOuLpCBmaXRfZ2xvYmFsKOybkOygkCwgbG9zcz0ibXNlIiwgc2VlZD1TRUVEKSDsmYAKICAgIGZpdF9nbG9iYWwo7JuQ7KCQLCBs"
           "b3NzPSJtYWUiLCBzZWVkPVNFRUQpIOulvCDrtojrn6wg7IaQ7IukIOyduOyekCDtlZjrgpjrp4wg64uk66W4IOuRkCDrqqjrjbjs"
           "nYQg66eM65Ok7Ja0IOykmC4KICAgIOybkOygkOqzvCDshpDsi6Trp4jri6QgYmVzdF9lcG9jaCwgTUFTRSjshozsiJgg7IWL7Ke4"
           "IOyekOumrCksIOq3uCDsm5DsoJDsnZgg6rOE7KCIIOuCmOydtOu4jCBNQVNFIOulvCDtlZwg7ZaJ7JSpIOywjeyWtCDspJguCiAg"
           "ICAi6rO17Zy07J28IOyXhuuKlCDso7wgTUFTRSDssKgoTVNFIOKIkiBNQUUpID0gKOyIq+yekCkiIOyZgCAi6rO17Zy07J28IOye"
           "iOuKlCDso7wgTUFTRSDssKgoTVNFIOKIkiBNQUUpID0gKOyIq+yekCkiIOulvAogICAgTUxQKE1TRSDshpDsi6QpIE1BU0Ug7JeQ"
           "7IScIE1MUChNQUUg7IaQ7IukKSBNQVNFIOulvCDruoAg6rCS7Jy866GcIOyGjOyImCDshYvsp7gg7J6Q66as6rmM7KeAIHByaW50"
           "IO2VtCDspJguCiAgICDqt7jrnpjtlITripQg7ZWcIOyepeyXkCDqt7jrprwg65GQIOqwnOulvCDrgpjrnoDtnogg6re466CkIOyk"
           "mC4g7Jm87Kq97J2AIOqzte2ctOydvCDsl4bripQg7KO8LCDsmKTrpbjsqr3snYAg6rO17Zy07J28IOyeiOuKlCDso7zsnbTqs6As"
           "IOq3uOumvOuniOuLpAogICAg6rOE7KCIIOuCmOydtOu4jCDCtyBNTFAoTVNFIOyGkOyLpCkgwrcgTUxQKE1BRSDshpDsi6QpIOun"
           "ieuMgCDshLgg6rCcLCDqt7jrprwg7KCc66qp7JeQIOybkOygkCDrgqDsp5wsIOqwgOuhnOy2lSDsnbTrpoQgIuuqqOuNuCIsIOyE"
           "uOuhnOy2lSDsnbTrpoQgIk1BU0UiIOulvCDri6zslYQg7KSYLgoKICAgIHJlcyA9IHt9CiAgICBmb3Igb3JpZ2luIGluIChPUklH"
           "SU5fUExBSU4sIE9SSUdJTl9IT0xJREFZKToKICAgICAgICBmb3IgbG9zcyBpbiAoIm1zZSIsICJtYWUiKToKICAgICAgICAgICAg"
           "ciA9IGZpdF9nbG9iYWwob3JpZ2luLCBsb3NzPWxvc3MsIHNlZWQ9U0VFRCwgZGV2aWNlPURFVklDRSkgICAjIOuRkCDrqqjrjbjs"
           "nYAgbG9zcyDsnbjsnpAg7ZWY64KY66eMIOuLpOumheuLiOuLpAogICAgICAgICAgICByZXNbKG9yaWdpbiwgbG9zcyldID0gcgog"
           "ICAgICAgICAgICBwcmludChmIuybkOygkCB7b3JpZ2lufSDCtyBNTFAoe2xvc3MudXBwZXIoKX0g7IaQ7IukKSDCtyDrs7Xsm5Dt"
           "lZwg7JeQ7Y+tIHtyWydiZXN0X2Vwb2NoJ119IMK3IE1BU0Uge3JbJ21hc2UnXTouM2Z9IgogICAgICAgICAgICAgICAgICBmIiDC"
           "tyDqs4TsoIgg64KY7J2067iMIE1BU0Uge1NOQUlWRV9UV09bb3JpZ2luXTouM2Z9IikKCiAgICBkX3BsYWluID0gcmVzWyhPUklH"
           "SU5fUExBSU4sICJtc2UiKV1bIm1hc2UiXSAtIHJlc1soT1JJR0lOX1BMQUlOLCAibWFlIildWyJtYXNlIl0KICAgIGRfaG9sID0g"
           "cmVzWyhPUklHSU5fSE9MSURBWSwgIm1zZSIpXVsibWFzZSJdIC0gcmVzWyhPUklHSU5fSE9MSURBWSwgIm1hZSIpXVsibWFzZSJd"
           "CiAgICBwcmludChmIuqzte2ctOydvCDsl4bripQg7KO8IE1BU0Ug7LCoKE1TRSDiiJIgTUFFKSA9IHtkX3BsYWluOi4zZn0iKQog"
           "ICAgcHJpbnQoZiLqs7XtnLTsnbwg7J6I64qUIOyjvCBNQVNFIOywqChNU0Ug4oiSIE1BRSkgPSB7ZF9ob2w6LjNmfSIpCgogICAg"
           "ZmlnLCBheGVzID0gcGx0LnN1YnBsb3RzKDEsIDIsIGZpZ3NpemU9KEZJR19XLCAzLjYpKQogICAgbmFtZXMgPSBbIuqzhOygiCDr"
           "gpjsnbTruIwiLCAiTUxQKE1TRSDshpDsi6QpIiwgIk1MUChNQUUg7IaQ7IukKSJdCiAgICBmb3IgYXgsIG9yaWdpbiwgdGl0bGUg"
           "aW4gemlwKGF4ZXMsIChPUklHSU5fUExBSU4sIE9SSUdJTl9IT0xJREFZKSwgKCLqs7XtnLTsnbwg7JeG64qUIOyjvCIsICLqs7Xt"
           "nLTsnbwg7J6I64qUIOyjvCIpKToKICAgICAgICB2YWxzID0gW1NOQUlWRV9UV09bb3JpZ2luXSwgcmVzWyhvcmlnaW4sICJtc2Ui"
           "KV1bIm1hc2UiXSwgcmVzWyhvcmlnaW4sICJtYWUiKV1bIm1hc2UiXV0KICAgICAgICBheC5iYXIobmFtZXMsIHZhbHMsIGNvbG9y"
           "PVtDT0xbIuyXsO2ajOyDiSJdLCBDT0xbIu2ajOyDiSJdLCBDT0xbIu2VmOuKmOyDiSJdXSkKICAgICAgICBmb3IgaywgdiBpbiBl"
           "bnVtZXJhdGUodmFscyk6CiAgICAgICAgICAgIGF4LnRleHQoaywgdiwgZiJ7djouM2Z9IiwgaGE9ImNlbnRlciIsIHZhPSJib3R0"
           "b20iKQogICAgICAgIGF4LnNldF95bGltKDAsIG1heCh2YWxzKSAqIDEuMTUpCiAgICAgICAgYXguc2V0X3RpdGxlKGYi7JuQ7KCQ"
           "IHtvcmlnaW59KHt0aXRsZX0pIikKICAgICAgICBheC5zZXRfeGxhYmVsKCLrqqjrjbgiKQogICAgICAgIGF4LnNldF95bGFiZWwo"
           "Ik1BU0UiKQogICAgZmlnLnRpZ2h0X2xheW91dCgpCiAgICBwbHQuc2hvdygpCgrtlbTshJ0KCi0g7JiI7IOBIOyFiyDqsIDsmrTr"
           "jbAg64u17J2AIOuRmOynuCwg44CM6rO17Zy07J28IOyXhuuKlCDso7zripQg64Ku7JWE7KeA6rOgIOqzte2ctOydvCDsnojripQg"
           "7KO864qUIOuGkuyVhOynhOuLpOOAjeyeheuLiOuLpC4g7JWE656YIOqwkuydgCDsnbQg6rWQ7J6s66W8IOunjOuToCDtmZjqsr0o"
           "66elIENQVSnsnZgg6rKw6rO87J6F64uI64ukLiDsm5DsoJAgMjAyNS0xMi0xMCjqs7XtnLTsnbwg7JeG64qUIOyjvCnsnYAgTUxQ"
           "KE1TRSDshpDsi6QpIDAuNTM2IMK3IE1MUChNQUUg7IaQ7IukKSAwLjMyNiDsnbTrnbwg7LCo6rCAIDAuMjEwIOyeheuLiOuLpC4g"
           "7JuQ7KCQIDIwMjUtMTAtMDEo6rO17Zy07J28IOyeiOuKlCDso7wp7J2AIE1MUChNU0Ug7IaQ7IukKSA0LjA0NyDCtyBNTFAoTUFF"
           "IOyGkOyLpCkgNC4zMjIg6528IOywqOqwgCDiiJIwLjI3NiDsnoXri4jri6QuIOuwmOyYrOumvCDsoIQg6rCS7Jy866GcIOu6gCDs"
           "sKjrnbwg67O07J2064qUIOuRkCDsiJjsnZgg7LCoIOKIkjAuMjc1IOyZgCDshozsiJgg7IWL7Ke4IOyekOumrOqwgCDri6TrpoXr"
           "i4jri6QuCi0g7YyQ64uoIOq3vOqxsDog6rO17Zy07J28IOyXhuuKlCDso7zsl5DshJwgTUxQKE1BRSDshpDsi6QpIDAuMzI2IOyd"
           "gCDqs4TsoIgg64KY7J2067iMIDAuNDE5IOuztOuLpOuPhCDrgq7sirXri4jri6QuIOqzte2ctOydvCDsnojripQg7KO87JeQ7ISc"
           "64qUIOyEuCDrp4nrjIDqsIAg66qo65GQIDQg67O064ukIO2BveuLiOuLpCjqs4TsoIgg64KY7J2067iMIDQuMzg4KS4g7J6F66Cl"
           "IDI47J28IOyWtOuUlOyXkOuPhCDqs7XtnLTsnbwg7Jes67aA6rCAIOyXhuyWtOyEnCDrkZAg66qo6424IOuqqOuRkCDsl7DtnLTr"
           "pbwg7YGs6rKMIOu5l+uCmOqwlOyKteuLiOuLpC4KLSDquYzri606IOybkOygkCAyMDI2LTA2LTI0IOydmCDtlZnsirUg7JyI64+E"
           "7JeQ7IScIOyghOyytCDrqqntkZzqsJLsnZggMy44JSDsnbgg7Y+J7J28IOqzte2ctOydvCDrqqntkZzqsJLsnbQgTUxQKE1TRSDs"
           "hpDsi6QpIOyGkOyLpOydmCA1OS43JSwgTUxQKE1BRSDshpDsi6QpIOyGkOyLpOydmCAzMi42JSDrpbwg7LCo7KeA7ZaI7Iq164uI"
           "64ukLiDsoJzqs7Eg7Jik7LCo64qUIOuTnOusvOqyjCDtgazqsowg67mX64KY6rCEIOuCoOydhCDrrLTqsoHqsowg67CY7JiB7ZWY"
           "66+A66GcLCBNTFAoTVNFIOyGkOyLpCnsnYAg6re4IOuCoOuTpOydhCDrp57stpTripQg7Kq97Jy866GcIO2VmeyKteuQmOqzoCDq"
           "t7jrp4ztgbwg6rO17Zy07J287J20IOyVhOuLjCDrgqDsnYQg64Ku6rKMIOyYiOy4oe2VqeuLiOuLpC4KLSDtnZTtlZwg7Iuk7IiY"
           "OiDssKjrpbwgTUFFIOKIkiBNU0Ug7Iic7ISc66GcIOu5vOyEnCDrtoDtmLjrpbwg6rGw6r6466GcIOydveuKlCDqsoMsIO2VnCDs"
           "m5DsoJDrp4wg67O06rOgIOOAjE1MUChNQUUg7IaQ7IukKeydtCDrqqjrk6Ag7KO87JeQIOuCq+uLpOOAjeqzoCDsoIHripQg6rKD"
           "7J6F64uI64ukLiDqsrDroaDsnYAg65GQIOybkOygkOydhCDrqqjrkZAg66eQ7ZW07JW8IO2VqeuLiOuLpC4KLSDsnbTrsogg7KO8"
           "7JeQIO2VoCDsnbw6IO2PieqwgO2RnOyXkCBNTFAoTUFFIOyGkOyLpCnsnZggTUFTRSDrpbwg642U7ZWY6rOgLCDrkZAg66y27J2M"
           "7J2YIO2Pieq3oCBNQVNFIOulvCDrlLDroZwg7KCB7Iq164uI64ukLiDtj4nsnbwg6rO17Zy07J287J20IOyeiOuKlCDso7zripQg"
           "7J20IOuqqOuNuOuhnCDsnbjroKXsnYQg7KCV7ZWY7KeAIOyViuuKlOuLpOqzoCDtj4nqsIDtkZwg7JWE656Y7JeQIO2VnCDspIQg"
           "64Ko6rmB64uI64ukLiDrgrQg7ZmU66m07J2YIOqwkuydgCDsobDquIgg64uk66W8IOyImCDsnojsirXri4jri6QuIENvbGFiIFQ0"
           "IOyXkOyEnCDsnrAg65GQIOywqOuKlCAwLjE5NyDqs7wg4oiSMC4yODYg7Jy866GcIOu2gO2YuOqwgCDqsJnslZjqs6AsIOyxhOyg"
           "kOuPhCDrkZAg7LCo7J2YIOu2gO2YuOyZgCDrjIDrnrXsnZgg7YGs6riw66eMIOu0heuLiOuLpC4KCuy9lOuTnOqwgCDri7XsnYQg"
           "66eM65Oc64qUIOuwqeyLnTogYGZpdF9nbG9iYWxgIOydgCDtkZzspIDtmZQg7Ya16rOEIMK3IO2VmeyKtSDsnIjrj4Qgwrcg6rKA"
           "7KadIOq1rOqwhCDCtyDsi5zrk5wgwrcg7KGw6riwIOyiheujjOulvCDrqqjrkZAg6rCZ6rKMIOuRkOqzoCwgYGxvc3NgIOyduOye"
           "kOuhnCDshpDsi6Qg7ZWo7IiY66eMIGBubi5NU0VMb3NzYCDsmYAgYG5uLkwxTG9zc2Ag6rCA7Jq0642wIO2VmOuCmOuhnCDsoJXt"
           "lanri4jri6QuIOq3uOuemOyEnCDrkZAg66qo64247J2YIOywqOydtOuKlCDshpDsi6Qg7ZWY64KY7JeQ7ISc66eMIOuCmOyYteuL"
           "iOuLpC4g7KGw6riwIOyiheujjOydmCDqsoDspp0g7IaQ7Iuk64+EIOqwmeydgCDshpDsi6TroZwg6rOE7IKw7ZWY66+A66GcLCDr"
           "s7Xsm5DtlZwg7JeQ7Y+t7J2AIOuRkCDrqqjrjbjsnbQg7ISc66GcIOuLpOulvCDsiJgg7J6I7Iq164uI64ukLgo="),
    "m2": ("8J+UkyDrr7jshZggMiDsmYTshLHrs7g6IEFJIOy9lOuTnCDspIQg7JWE656Y7JeQIOuTpOyWtOyZgOyVvCDtlaAg7L2U65Oc64qU"
           "IOydtCDsoJXrj4TsnoXri4jri6QKCuKWoCDsnbQg7L2U65Oc66W8IOu2gOuluCDtlITroaztlITtirgg7JiICgogICAg7JWE656Y"
           "IOyghOyehCDri7Tri7nsnpAg7L2U65Oc7JeQ7IScIOyeheugpSAyOOydvOydhCDsnpDrpbTripQg7KSE6rO8IOyeheugpSDsnIjr"
           "j4Qg64Kg7Kec66W8IOywjeuKlCDspITsnYQg6rOg7LOQIOykmC4g7KeA6riI7J2AIOybkOygkCDri7nsnbzquYzsp4Ag65Ok7Ja0"
           "6rCA64qU642wLAogICAg7JuQ7KCQ66eI64ukIOybkOygkCAyOOydvCDsoITrtoDthLAg7JuQ7KCQIOyghOuCoOq5jOyngCAyOOyd"
           "vOunjCDrk6TslrTqsIDqsowgTUFUIOulvCDsnpjrnbwg7KSYLgogICAg7J2066+4IOyeiOuKlCBNQVQsIERBVEVTLCBPUklHSU5T"
           "LCBQTEFJTl9PUklHSU5TLCBIT0xfT1JJR0lOUywgTCwgSCwgcWRlbl9vZiwgc2NvcmUsIG1scF9mb3JlY2FzdCDripQKICAgIOq3"
           "uOuMgOuhnCDsk7Dqs6Ag64uk7IucIOunjOuTpOyngCDrp5DslYQg7KSYLiDsm5DsoJDrp4jri6Qg7J6F66ClIOyciOuPhCDssqvr"
           "gqDqs7wg64Gd64KgLCDrgZ3rgqDsl5DshJwg7JuQ7KCQ6rmM7KeAIOupsOy5oOyduOyngCwgTUFTRSDrpbwg7ZWcIO2WieyUqSDs"
           "sI3qs6AKICAgICLsm5DsoJAg7KCE64Kg7JeQ7IScIOuBneuCmOuKlCDsnoXroKUg7JyI64+EIOyImCA9ICjsiKvsnpApIiDsmYAg"
           "Iuqzte2ctOydvCDsl4bripQg65GQIOyjvCDtj4nqt6AgTUFTRSA9ICjsiKvsnpApIiDrpbwg7ZWcIO2WieyUqSBwcmludCDtlbQg"
           "7KSYLgogICAgTUFTRSDripQg7IaM7IiYIOyFi+ynuCDsnpDrpqzroZwg7LCN7Ja0IOykmC4gIu2BrOqyjCDruZfrgpjqsJHri4jr"
           "i6QiIOykhOydgCDqs7XtnLTsnbwg7J6I64qUIOyEuCDso7wg7Y+J6reg6rO8IOybkOygkCA16rCcIO2Pieq3oCBNQVNFIOulvAog"
           "ICAg7Iir7J6Q66Gc66eMIOywjeuKlCDspITroZwg67CU6r646rOgLCAi7YyQ64uoIDog7Iug6rK966ed7J2AIOuyhOumveuLiOuL"
           "pCIg7KSE7J2AIOyngOybjCDspJguCiAgICDqt7jrnpjtlITripQg6rCA66Gc7LaV7JeQIOybkOygkCA16rCcLCDshLjroZzstpXs"
           "l5AgTUFTRSDrpbwg65GUIOunieuMgCA16rCcIO2VnCDsnqXsnLzroZwg6re466as6rOgIOqwgOuhnOy2lSDsnbTrpoQgIuybkOyg"
           "kCIsCiAgICDshLjroZzstpUg7J2066aEICJNQVNFIiDrpbwg64us7JWEIOykmC4KCiAgICBtYXNlX2J5LCBnYXBfYnkgPSB7fSwg"
           "e30KICAgIGZvciBvcmlnaW4gaW4gT1JJR0lOUzoKICAgICAgICBvID0gREFURVMuZ2V0X2xvYyhvcmlnaW4pCiAgICAgICAgcmF3"
           "MjggPSBNQVRbOiwgbyAtIEw6b10gICAgICAgICAgICAgICAgICAgIyDsm5DsoJAgMjjsnbwg7KCEIH4g7JuQ7KCQIOyghOuCoC4g"
           "64GdIOuyiO2YuCBvKOybkOygkCDri7nsnbwp64qUIOuTpOyWtOqwgOyngCDslYrripTri6QKICAgICAgICBwcmVkID0gbWxwX2Zv"
           "cmVjYXN0KG9yaWdpbiwgcmF3MjgpCiAgICAgICAgbWFzZV9ieVtvcmlnaW5dID0gc2NvcmUoTUFUWzosIG86byArIEhdLCBwcmVk"
           "LCBxZGVuX29mKG9yaWdpbikpWyJtYXNlIl0KICAgICAgICBmaXJzdCwgbGFzdCA9IERBVEVTW28gLSBMXSwgREFURVNbbyAtIDFd"
           "CiAgICAgICAgZ2FwX2J5W29yaWdpbl0gPSAob3JpZ2luIC0gbGFzdCkuZGF5cyAgICAgIyDsnoXroKUg64Gd64Kg7JeQ7IScIOyb"
           "kOygkOq5jOyngCDrqbDsuaDsnbjsp4AKICAgICAgICBwcmludChmIuybkOygkCB7b3JpZ2luLmRhdGUoKX0gfCDsnoXroKUg7JyI"
           "64+EIHtmaXJzdC5kYXRlKCl9IH4ge2xhc3QuZGF0ZSgpfSIKICAgICAgICAgICAgICBmIiB8IOuBneuCoOyXkOyEnCDsm5DsoJDq"
           "uYzsp4Age2dhcF9ieVtvcmlnaW5dfeydvCB8IE1BU0Uge21hc2VfYnlbb3JpZ2luXTouM2Z9IikKICAgIHByaW50KGYi7JuQ7KCQ"
           "IOyghOuCoOyXkOyEnCDrgZ3rgpjripQg7J6F66ClIOyciOuPhCDsiJggPSB7c3VtKGcgPT0gMSBmb3IgZyBpbiBnYXBfYnkudmFs"
           "dWVzKCkpfSIpCiAgICBwcmludChmIuqzte2ctOydvCDsl4bripQg65GQIOyjvCDtj4nqt6AgTUFTRSA9IHtucC5tZWFuKFttYXNl"
           "X2J5W29dIGZvciBvIGluIFBMQUlOX09SSUdJTlNdKTouM2Z9IikKICAgIHByaW50KGYiICDqs7XtnLTsnbwg7J6I64qUIOyEuCDs"
           "o7wg7Y+J6regIE1BU0Uge25wLm1lYW4oW21hc2VfYnlbb10gZm9yIG8gaW4gSE9MX09SSUdJTlNdKTouM2Z9IgogICAgICAgICAg"
           "ZiIgwrcg7JuQ7KCQIDXqsJwg7Y+J6regIE1BU0Uge25wLm1lYW4obGlzdChtYXNlX2J5LnZhbHVlcygpKSk6LjNmfSIpCgogICAg"
           "ZmlnLCBheCA9IHBsdC5zdWJwbG90cyhmaWdzaXplPShGSUdfVyAqIDAuOCwgMy42KSkKICAgIGF4LmJhcihbc3RyKG8uZGF0ZSgp"
           "KSBmb3IgbyBpbiBPUklHSU5TXSwgbGlzdChtYXNlX2J5LnZhbHVlcygpKSwgY29sb3I9Q09MWyLrgqjsg4kiXSwgd2lkdGg9MC42"
           "KQogICAgYXguc2V0X3hsYWJlbCgi7JuQ7KCQIikKICAgIGF4LnNldF95bGFiZWwoIk1BU0UiKQogICAgcGx0LnNob3coKQoK7ZW0"
           "7ISdCgotIO2LgOumsCDqs7Mg7LC+6riw7J2YIOuLteydgCDssqvsp7gg67O06riwIGBNQVRbOiwgbyAtIDI3Om8gKyAxXWAg7J6F"
           "64uI64ukLiDtjIzsnbTsjawg7J6Q66W06riw64qUIOuBnSDrsojtmLgg7Je07J2EIO2PrO2VqO2VmOyngCDslYrsnLzrr4DroZwg"
           "7J20IOykhOydgCDsm5DsoJAgMjfsnbwg7KCE67aA7YSwIOybkOygkCDri7nsnbzquYzsp4AgMjjsnbzsnYQg6rCA7KC47Ji164uI"
           "64ukLiDsm5DsoJAg64u57J287J2YIOyKueywqOuKlCDssYTsoJAg7LKr64Kg7J2YIOygleuLteyeheuLiOuLpC4g65GY7Ke4IOuz"
           "tOq4sOydmCDsl63rs4Qg7Y+J6regwrftkZzspIDtjrjssKjsmYAg7IWL7Ke4IOuztOq4sOydmCBNQVNFIOu2hOuqqOuKlCDrqqjr"
           "kZAg7JuQ7KCQ67O064ukIOyVnuyEoCDsnpDro4zroZwg7J6wIOqwkuydtOudvCDqt7jrjIDroZwg65Gh64uI64ukLgotIO2MkOuL"
           "qCDqt7zqsbA6IOqzoOy5nCDrkqQg7JuQ7KCQIDXqsJwg66qo65GQIOOAjOuBneuCoOyXkOyEnCDsm5DsoJDquYzsp4AgMeydvOOA"
           "jeydtOqzoCwg7J20IOq1kOyerOulvCDrp4zrk6Ag7ZmY6rK97JeQ7IScIOqzte2ctOydvCDsl4bripQg65GQIOyjvCDtj4nqt6Ag"
           "TUFTRSDripQgMC43Mzkg66GcLCDtkZwgMSDsnZgg6rOE7KCIIOuCmOydtOu4jCAwLjQ0NSDrs7Tri6Qg7YG964uI64ukLiDsm5Ds"
           "oJAgNeqwnCDtj4nqt6DsnYAgMS44NDgg7J6F64uI64ukLiDrgrQg7ZmU66m07J2YIOqwkuydgCDshozsiJgg7IWL7Ke4IOyekOum"
           "rOyXkOyEnCDsobDquIgg64uk66W8IOyImCDsnojsirXri4jri6QuCi0g64iE7IiY6rCAIOyeiOuKlCDsoITsnoTsnpAg7L2U65Oc"
           "64qUIOygkOyImOulvCDrgpjsgZjqsowg66eM65Ok7JeI7Iq164uI64ukLiDqs7XtnLTsnbwg7JeG64qUIOuRkCDso7wg7Y+J6reg"
           "7J2AIDIuMTM0LCDsm5DsoJAgNeqwnCDtj4nqt6DsnYAgMi43NzAg7J6F64uI64ukLiDqsIDspJHsuZjripQg7J6F66ClIOuBneuC"
           "oOydmCDri6TsnYwg64Kg67aA7YSwIDfsnbzsnYQg64K064+E66GdIO2VmeyKteuQkOyKteuLiOuLpC4g7J6F66Cl7J20IOybkOyg"
           "kCDri7nsnbzsl5DshJwg64Gd64KY66m0IOuqqOuNuOydgCDsm5DsoJAg64uk7J2MIOuCoOu2gO2EsOydmCA37J28IOuqqOyWkeyd"
           "hCDrgrTripTrjbAsIOyxhOygkOydgCDsm5DsoJAg64u57J2867aA7YSw6528IOyalOydvOydtCDtlZjro6jslKkg7Ja06riL64Kp"
           "64uI64ukLgotIO2dlO2VnCDsi6TsiJg6IOygkOyImOqwgCDrgpjruaDsoYzsnLzri4gg64iE7IiY6rCAIOyVhOuLiOudvOqzoCDs"
           "nb3ripQg6rKD7J6F64uI64ukLiDsm5DsoJAgMjAyNS0xMC0wMSDrp4zsnYAg64iE7IiY6rCAIOyeiOuKlCDsoITsnoTsnpAg7L2U"
           "65Oc7J2YIE1BU0UoNC4wMTUp6rCAIOqzoOy5nCDsvZTrk5zsnZggTUFTRSg0LjA0Nynrs7Tri6Qg7Jik7Z6I66CkIOyekeyKteuL"
           "iOuLpC4g64iE7IiY64qUIOygkOyImOulvCDsoovqsozrj4Qg64KY7IGY6rKM64+EIOunjOuTpOyWtCDsoJDsiJjroZzripQg7JWM"
           "7JWE7LGMIOyImCDsl4bqs6AsIOyeheugpSDsnIjrj4TsnZgg64Kg7Kec66GcIO2ZleyduO2VqeuLiOuLpC4g7JuQ7KCQIO2VmOuC"
           "mOunjCDqs6DsuZjrqbQg7J6F66ClIOyciOuPhCDsiJjqsIAgMSDsnbTrnbwg7LGE7KCQ7J20IOKdjCDrpbwg64OF64uI64ukLiDs"
           "npDrpbTripQg7ZaJ66eMIOqzoOy5mOqzoCDrgqDsp5zrpbwg7LCN64qUIO2WieydhCDsmJsg67KI7Zi466GcIOuRkOuptCDrgZ3r"
           "gqDsl5DshJwg7JuQ7KCQ6rmM7KeAIDDsnbzsnbQg7LCN7ZiAIOyeheugpSDsnIjrj4Qg7IiY6rCAIDAg7J6F64uI64ukLgotIOyd"
           "tOuyiCDso7zsl5Ag7ZWgIOydvDog6rOg7LmcIOy9lOuTnOydmCBNTFAoTVNFIOyGkOyLpCnrj4Qg6rO17Zy07J28IOyXhuuKlCDr"
           "kZAg7KO87JeQ7IScIOqzhOygiCDrgpjsnbTruIzrs7Tri6QgTUFTRSDqsIAg64aS7Jy866+A66GcLCDsnbQg66qo64247J2AIO2P"
           "ieqwgO2RnOyXkCDsoIHqs6Ag7Yis7J6FIO2bhOuztOuhnOuKlCDsmKzrpqzsp4Ag7JWK7Iq164uI64ukLiDshpDsi6TsnYAg66+4"
           "7IWYIDEg7J2YIE1MUChNQUUg7IaQ7IukKeyymOufvCBNQUUg66GcIOuRkOqzoCwg7IOIIOuqqOuNuOydhCDtj4nqsIDtkZzsl5Ag"
           "642U7ZWgIOuVjOuniOuLpCDsm5DsoJDrs4Qg7J6F66ClIOyciOuPhOydmCDrgZ3rgqDsnYQg7ZWcIO2WieyUqSDtlajqu5gg7Lac"
           "66Cl7ZW0IOuRoeuLiOuLpC4KCuy9lOuTnOqwgCDri7XsnYQg66eM65Oc64qUIOuwqeyLnTogYE1BVFs6LCBvIC0gTDpvXWAg64qU"
           "IG/iiJIyOCDrsojsp7gg7Je067aA7YSwIG/iiJIxIOuyiOynuCDsl7TquYzsp4AgMjjsnbzsnbTrnbwg66eI7KeA66eJIOyXtOyd"
           "tCDsm5DsoJAg7KCE64Kg7J6F64uI64ukLiBgREFURVNbbyAtIDFdYCDsnYQg64Gd64Kg66GcIOywjeqzoCDrgZ3rgqDsl5DshJwg"
           "7JuQ7KCQ6rmM7KeAIOupsOy5oOyduOyngCDqs4TsgrDtlZjrqbQg64iE7IiY6rCAIOuCoOynnOuhnCDrk5zrn6zrgqnri4jri6Qu"
           "IOyYiOy4oeydgCBgbWxwX2ZvcmVjYXN0YCDqsIAgYERFVklDRWAg7J2YIOuqqOuNuOuhnCDqs4TsgrDtlbQgYC5jcHUoKWAg66Gc"
           "IOqwgOyguOyYpOuvgOuhnCwg6rOg7LmcIOy9lOuTnOuKlCDsnpDrpbTripQg7ZWcIO2WieqzvCDrgqDsp5wg7Lac66Cl67+Q7J6F"
           "64uI64ukLgo="),
}
# 항목별 힌트 — base64(JSON)
_HINT = _dec_json(
               "eyJtMSI6eyLqs7XtnLTsnbwg7JeG64qUIOyjvCBNQVNFIOywqChNU0Ug4oiSIE1BRSkiOlsiYGZpdF9nbG9iYWwoT1JJR0lOX1BM"
               "QUlOLCBsb3NzPVwibXNlXCIpYCDsnZggYG1hc2VgIOyXkOyEnCBgZml0X2dsb2JhbChPUklHSU5fUExBSU4sIGxvc3M9XCJtYWVc"
               "IilgIOydmCBgbWFzZWAg66W8IOu6gCDqsJLsnoXri4jri6QuIOu5vOuKlCDsiJzshJzqsIAgTVNFIOKIkiBNQUUg7J247KeALCDr"
               "kZAg7Zi47Lac7J2YIOyLnOuTnOyZgCBERVZJQ0Ug6rCAIOqwmeydgOyngCDrs7TshLjsmpQuIOOAjOqzte2ctOydvCDsl4bripQg"
               "7KO8IE1BU0Ug7LCoKE1TRSDiiJIgTUFFKSA9IOyIq+yekOOAjSDtlZwg7KSE66GcIOy2nOugpe2VqeuLiOuLpC4g44CMPeOAjSDr"
               "kqTsl5DripQg7LCoIO2VmOuCmOunjCDsoIHsirXri4jri6Qo44CM4oCmID0gMC41IOKIkiAwLjMgPSAwLjLjgI0g7LKY65+8IOyL"
               "neydhCDri6Qg7KCB7Jy866m0IOyyqyDsiKvsnpAgMC41IOulvCDsnb3sirXri4jri6QpIiwi64K0IOywqOqwgCDslrTripAg7JuQ"
               "7KCQ7J2YIOyWtOuKkCDrkZAg66qo64247J2EIOyWtOuKkCDsiJzshJzroZwg67qAIOqwkuyduOyngCJdLCLqs7XtnLTsnbwg7J6I"
               "64qUIOyjvCBNQVNFIOywqChNU0Ug4oiSIE1BRSkiOlsiYE9SSUdJTl9IT0xJREFZYCDsl5DshJwg6rCZ7J2AIOyInOyEnChNTFAo"
               "TVNFIOyGkOyLpCkg4oiSIE1MUChNQUUg7IaQ7IukKSnroZwg67qAIOqwkuyeheuLiOuLpC4g65GQIOybkOygkOydhCDtlZwg67CY"
               "67O17JeQ7IScIOyymOumrO2VoCDrlYwg7JuQ7KCQIOydtOumhOydhCDrsJTqv5Qg7KCB7KeAIOyViuyVmOuKlOyngCwg44CM6rO1"
               "7Zy07J28IOyeiOuKlCDso7wgTUFTRSDssKgoTVNFIOKIkiBNQUUpID0g7Iir7J6Q44CNIO2VnCDspITroZwg7Lac66Cl7ZaI64qU"
               "7KeAIOuztOyEuOyalC4g44CMPeOAjSDrkqTsl5DripQg7LCoIO2VmOuCmOunjCDsoIHsirXri4jri6Qo44CM4oCmID0gMC41IOKI"
               "kiAwLjMgPSAwLjLjgI0g7LKY65+8IOyLneydhCDri6Qg7KCB7Jy866m0IOyyqyDsiKvsnpAgMC41IOulvCDsnb3sirXri4jri6Qp"
               "Iiwi64K0IOywqOqwgCDsm5DsoJAgMjAyNS0xMC0wMSjqs7XtnLTsnbwg7J6I64qUIOyjvCnsnZgg65GQIOuqqOuNuOyXkOyEnCDr"
               "gpjsmKgg6rCS7J247KeAIl0sInBsb3QiOlsi6re4656Y7ZSE66W8IOydtCDshYDsl5DshJwg6re466C464qU7KeALCDqsIDroZzs"
               "tpXqs7wg7IS466Gc7LaV7JeQIOydtOumhCjjgIzrqqjrjbjjgI3Ct+OAjE1BU0XjgI0p7J2EIOuLrOyVmOuKlOyngCDrs7TshLjs"
               "mpQuIOq3uOuemO2UhOuKlCDqt7jrprzsnbQg7J6I64qU7KeA7JmAIOuRkCDstpXsnZgg7J2066aE66eMIOyxhOygkO2VqeuLiOuL"
               "pCIsIuuCtCDqt7jrnpjtlITsnZgg6rCA66Gc7LaVwrfshLjroZzstpUg7J2066aE7J20IOyWtOuUlOyEnCDruaDsoYzripTsp4Ai"
               "XSwiZXJyb3IiOlsi7Jik66WYIOusuOyepSDssqsg7ZaJ7J2EIOq3uOuMgOuhnCDtipzthLDsl5Ag67O07JesIOyjvOqzoCwg44CM"
               "66+47IWYIDEg7KSA67mEIOyFgOOAjeydtCDrp4zrk6Ag7J2066aEKGZpdF9nbG9iYWwgwrcgT1JJR0lOX1BMQUlOIMK3IE9SSUdJ"
               "Tl9IT0xJREFZIMK3IFNOQUlWRV9UV08gwrcgU0VFRCDCtyBERVZJQ0Up6rO8IOyEpOy5mCDshYDsnbQg66eM65OgIOydtOumhCjq"
               "t7jrprwg7Y+tIGBGSUdfV2AsIOyDiSDsgqzsoIQgYENPTGAp7J2AIOuLpOyLnCDrp4zrk6Tsp4Ag66eQ6rOgIOq4gOyekCDqt7jr"
               "jIDroZwg7JOw65286rOgIOu2hOuqhe2eiCDsoIHsnLzshLjsmpQuIOydtCDsnbTrpoTsnbQg7JeG64uk64qUIOyYpOulmOuptCDr"
               "p6gg7JyEIOyEpOy5mCDshYDqs7wg44CM66+47IWYIDEg7KSA67mEIOyFgOOAjeu2gO2EsCDri6Tsi5wg7Iuk7ZaJ7ZWp64uI64uk"
               "LiBgZml0X2dsb2JhbGAg7J2YIGBsb3NzYCDripQgXCJtc2VcIiDsmYAgXCJtYWVcIiDrkZAg66y47J6Q7Je066eMIOuwm+yKteuL"
               "iOuLpCIsIuydtCDsmKTrpZjqsIAg7Ja064qQIO2WieyXkOyEnCDsmZwg64Ks64qU7KeAIl19LCJtMiI6eyLsm5DsoJAg7KCE64Kg"
               "7JeQ7IScIOuBneuCmOuKlCDsnoXroKUg7JyI64+EIOyImCI6WyLsm5DsoJDrp4jri6Qg7J6F66ClIOyciOuPhOydmCDrgZ3rgqDs"
               "l5DshJwg7JuQ7KCQ6rmM7KeAIOupsOy5oOyduOyngCDstpzroKXtlbQgMeydvOyduOyngCDrs7TshLjsmpQuIOuCoOynnOulvCDs"
               "sI3ripQg7ZaJKGBmaXJzdCwgbGFzdCA9IOKApmAp64+EIOyekOuluCDrspTsnITsmYAg6rCZ7J2AIOuyiO2YuOuhnCDqs6Dss5Ds"
               "lbwg7Lac66Cl65CcIOuBneuCoOydtCDsi6TsoJwg7J6F66Cl7J2YIOuBneuCoOqzvCDqsJnslYTsp5Hri4jri6QuIOOAjOybkOyg"
               "kCDsoITrgqDsl5DshJwg64Gd64KY64qUIOyeheugpSDsnIjrj4Qg7IiYID0g7Iir7J6Q44CNIO2VnCDspITroZwg7Lac66Cl7ZWp"
               "64uI64ukIiwi7JuQ7KCQIOuLueydvOydmCDsirnssKjripQg7LGE7KCQIOyyq+uCoOydmCDsoJXri7XsnoXri4jri6QuIOyeheug"
               "pSAyOOydvOydgCDsm5DsoJAg7KCE64Kg7JeQ7IScIOuBneuCmOyVvCDtlbTshJwg64Gd64Kg7JeQ7IScIOybkOygkOq5jOyngCAx"
               "7J287J6F64uI64ukLiDtjIzsnbTsjawg7J6Q66W06riwIGBhOmJgIOuKlCBiIOuyiOynuCDsl7TsnYQg7Y+s7ZWo7ZWY7KeAIOyV"
               "iuyKteuLiOuLpC4g7J6Q66W064qUIO2WieunjCDqs6DsuZjqs6Ag64Kg7Kec66W8IOywjeuKlCDtlonsnYQg7JibIOuyiO2YuOuh"
               "nCDrkZDrqbQg64Gd64Kg7JeQ7IScIOybkOygkOq5jOyngCAw7J287J20IOywje2emeuLiOuLpCIsIuyeheugpSAyOOydvOydhCDs"
               "npDrpbTripQg7ZaJ6rO8IOuCoOynnOulvCDssI3ripQg7ZaJ7J20IOybkOygkOuniOuLpCDslrTripAg64Kg7JeQ7IScIOuBneuC"
               "mOuKlOyngCJdLCLqs7XtnLTsnbwg7JeG64qUIOuRkCDso7wg7Y+J6regIE1BU0UiOlsi7JuQ7KCQIDXqsJwg66qo65GQ7JeQ7ISc"
               "IOyeheugpSDsnIjrj4TsnZgg64Gd64Kg7J20IOybkOygkCDsoITrgqDsnbjsp4Ag7Lac66Cl65CcIOuCoOynnOuhnCDtmZXsnbjt"
               "lZjshLjsmpQuIO2Pieq3oOydgCDqs7XtnLTsnbwg7JeG64qUIOuRkCDso7zsnbggYFBMQUlOX09SSUdJTlNgIOydmCDrkZAg7JuQ"
               "7KCQKDIwMjUtMTItMTAgwrcgMjAyNi0wNi0yNCnsnLzroZzrp4wg64OF64uI64ukLiDjgIzqs7XtnLTsnbwg7JeG64qUIOuRkCDs"
               "o7wg7Y+J6regIE1BU0UgPSDsiKvsnpDjgI0g7ZWcIOykhOuhnCDstpzroKXtlanri4jri6QiLCLsm5DsoJAg67CY67O166y4IOyV"
               "iOydmCDsnpDrpbTripQg7ZaJIO2VmOuCmOulvCDqs6DsuZjrqbQg7JuQ7KCQIDXqsJzqsIAg7ZWo6ruYIOuwlOuAneuLiOuLpC4g"
               "7JuQ7KCQIO2VmOuCmOunjCDrlLDroZwg6rOg7LmY66m0IOuCmOuouOyngCDsm5DsoJDsl5DripQg64Gd64Kg7J20IOybkOygkCDr"
               "i7nsnbzsnbgg7JyI64+E6rCAIOq3uOuMgOuhnCDrgqjsirXri4jri6QiLCLqs7XtnLTsnbwg7JeG64qUIOuRkCDso7zsnZgg7JuQ"
               "7KCQ66eI64ukIOyeheugpSDsnIjrj4TqsIAg7Ja064qQIOuCoOyXkOyEnCDrgZ3rgpjripTsp4AiXSwicGxvdCI6WyLqt7jrnpjt"
               "lITrpbwg7J20IOyFgOyXkOyEnCDqt7jroLjripTsp4AsIOqwgOuhnOy2leqzvCDshLjroZzstpXsl5Ag7J2066aEKOOAjOybkOyg"
               "kOOAjcK344CMTUFTReOAjSnsnYQg64us7JWY64qU7KeAIOuztOyEuOyalC4g6re4656Y7ZSE64qUIOq3uOumvOydtCDsnojripTs"
               "p4DsmYAg65GQIOy2leydmCDsnbTrpoTrp4wg7LGE7KCQ7ZWp64uI64ukIiwiYGF4LnNldF94bGFiZWwoXCLsm5DsoJBcIilgIOqz"
               "vCBgYXguc2V0X3lsYWJlbChcIk1BU0VcIilgIOyymOufvCDrkZAg7LaV7J2YIOydtOumhOydhCDrqqjrkZAg7KCB7Jy865286rOg"
               "IFBST01QVCDsl5Ag7JOw7IS47JqULiDrp4nrjIDsnZgg7IiY7JmAIOuGkuydtOuKlCDssYTsoJDtlZjsp4Ag7JWK7Iq164uI64uk"
               "Iiwi64K0IOq3uOuemO2UhOydmCDqsIDroZzstpXCt+yEuOuhnOy2lSDsnbTrpoTsnbQg7Ja065SU7IScIOu5oOyhjOuKlOyngCJd"
               "LCJmb3JiaWQiOlsi7KCE7J6EIOuLtOuLueyekOydmCDrkZAg7ZaJIOOAjOKApiDrs7Tri6Qg7YGs6rKMIOu5l+uCmOqwkeuLiOuL"
               "pOOAjSDsmYAg44CM7YyQ64uoIDog7Iug6rK966ed7J2AIOuyhOumveuLiOuLpOOAjSDrpbwg7Lac66Cl7JeQ7IScIOyngOyasOud"
               "vOqzoCBQUk9NUFQg7JeQIOyggeycvOyEuOyalC4g65GYIOuLpCDsoJDsiJgg7ZWY64KY66GcIOuqqOuNuOydhCDrsoTrprDri6Tq"
               "s6Ag64uo7KCV7ZWcIOusuOyepeyeheuLiOuLpCIsIuyWtOuKkCBwcmludCDtlonsnbQg7J20IOuRkCDrrLjsnqXsnYQg7Lac66Cl"
               "7ZWY64qU7KeAIl0sImVycm9yIjpbIuyYpOulmCDrrLjsnqUg7LKrIO2WieydhCDqt7jrjIDroZwg7Yqc7YSw7JeQIOuztOyXrCDs"
               "o7zqs6AsIOOAjOuvuOyFmCAyIOykgOu5hCDshYDjgI3snbQg66eM65OgIOydtOumhChNQVQgwrcgREFURVMgwrcgT1JJR0lOUyDC"
               "tyBQTEFJTl9PUklHSU5TIMK3IEhPTF9PUklHSU5TIMK3IEwgwrcgSCDCtyBxZGVuX29mIMK3IHNjb3JlIMK3IERFVklDRSDCtyBN"
               "T0RFTFMgwrcgTVUgwrcgU0QgwrcgbWxwX2ZvcmVjYXN0KeqzvCDshKTsuZgg7IWA7J20IOunjOuToCDsnbTrpoQo6re466a8IO2P"
               "rSBgRklHX1dgLCDsg4kg7IKs7KCEIGBDT0xgKeydgCDri6Tsi5wg66eM65Ok7KeAIOunkOqzoCDquIDsnpAg6re464yA66GcIOyT"
               "sOudvOqzoCDrtoTrqoXtnogg7KCB7Jy87IS47JqULiDsnbQg7J2066aE7J20IOyXhuuLpOuKlCDsmKTrpZjrqbQg66eoIOychCDs"
               "hKTsuZgg7IWA6rO8IOOAjOuvuOyFmCAyIOykgOu5hCDshYDjgI3rtoDthLAg64uk7IucIOyLpO2Wie2VqeuLiOuLpC4gYG1scF9m"
               "b3JlY2FzdGAg7J2YIOyeheugpeydgCDsl63rp4jri6QgMjjsnbzsnoXri4jri6QiLCLsnbQg7Jik66WY6rCAIOyWtOuKkCDtlons"
               "l5DshJwg7JmcIOuCrOuKlOyngCJdfX0=")
_ASK = {"m1": "full", "m2": "blank"}   # 미션 1 은 그대로 보낼 질문, 미션 2 는 빈칸을 내가 채운다
_ASK_HEAD = {"full": "◼ 튜터에 그대로 보낼 질문 (코드는 고치지 말라고 분명히 적습니다)",
             "blank": "◼ 튜터에 보낼 질문 (빈칸은 내가 채웁니다)"}
_ASK_FORM = {"full": "내 PROMPT 는 「{prompt}」 였는데 「{name}」 채점이 ❌ 였습니다."
                     " AI 코드 표시선 아래 코드를 보고 「{ask}」 에 3~5문장으로 답해 주세요. 코드는 고치지 마세요.",
             "blank": "내 PROMPT 는 ____ 였는데 「{name}」 채점이 ❌ 였습니다. AI 코드 표시선 아래 코드에서"
                      " 「{ask}」 부터 짚고, 어느 행을 무엇으로 고칠지 한 문장으로 답해 주세요."}

_FIGS = []                                 # 「PROMPT 셀」이 만든 그림을 모으는 자리
_STATE = {"tag": None, "buf": None, "out": None, "ids0": set()}
_TRY = {}                                  # 미션마다 ❌ 가 몇 번째인지 — 2단 힌트가 이 수를 봅니다
_orig_figure = getattr(plt.figure, "_orig", plt.figure)   # 이 셀을 다시 실행해도 겹겹이 싸이지 않게


def _figure(*a, **k):
    f = _orig_figure(*a, **k)
    _FIGS.append(f)
    return f


_figure._orig = _orig_figure
plt.figure = _figure                       # subplots · pandas · seaborn 도 이 경로를 거칩니다


def _get(name):
    """이름이 살아 있으면 그 값, 없으면 None — 아직 만들지 않은 이름에도 채점이 죽지 않게"""
    return globals().get(name)


def _open_figs():
    """지금 열려 있는 그림들 — 래퍼를 거치지 않고 읽습니다"""
    try:
        return [plt._pylab_helpers.Gcf.figs[n].canvas.figure for n in plt.get_fignums()]
    except Exception:
        return []


def _nums(text):
    """출력 텍스트에서 숫자만 뽑습니다 (천 단위 쉼표 제거, 지수 표기 포함, 유니코드 마이너스 − 는 - 로 읽음)"""
    picked = []
    for seg in re.findall(r"-?\d[\d,]*\.?\d*(?:[eE][-+]?\d+)?", (text or "").replace("−", "-")):
        try:
            picked.append(float(seg.replace(",", "")))
        except ValueError:
            continue
    return picked


def _num_txt(v, fmt=",.6g"):
    """채점 문장에 찍을 수 — 음수 부호는 본문과 같은 − (U+2212) 로 씁니다"""
    return format(v, fmt).replace("-", "−")


def _value_lines(txt, units):
    """값 채점이 먼저 볼 행 — 단위가 붙은 행, 또는 「이름 = 값」 꼴의 행.

    이름 찾기는 이 행들에서 먼저 하고, 여기서 못 찾으면 출력의 모든 행(「이름: 값」 꼴 포함)에서 합니다.
    이름이 있는 행이 아예 없으면 숫자를 읽지 않고 ❌ 로 알립니다(아래 `judge`).
    """
    picked = []
    for line in (txt or "").splitlines():
        if not line.strip():
            continue
        if "=" in line or any(unit and unit in line for unit in units):
            picked.append(line)
    return picked


def _label_lines(val_lines, label):
    """라벨이 있는 값줄만 골라, 그 줄에서 라벨 뒤(「=」·「:」 뒤) 토막을 돌려줍니다.

    값줄의 숫자를 한 주머니에 담아 기준 범위와 대조하면 **다른 항목 행의 값**이 이 항목을
    헛통과시킵니다. 그래서 라벨이 있는 줄의 라벨 뒤만 봅니다. 라벨이 있는 줄이 없으면 빈 목록이고,
    부르는 쪽이 ❌ 로 알립니다.
    라벨 찾기는 세 단입니다 — 정확히 같은 문자열 → 공백·괄호를 걷어낸 꼴 → 숫자 표기까지 맞춘 꼴
    (「5.0」 과 「5」, 「3.50」 과 「3.5」 를 같게 봅니다).
    """
    def _tighten(txt):                     # 공백과 괄호(안의 말까지) 를 걷어낸 글자 · 원래 자리
        keep, depth = [], 0
        for i, ch in enumerate(txt or ""):
            if ch in "([":
                depth += 1
            elif ch in ")]":
                depth = max(0, depth - 1)
            elif depth == 0 and not ch.isspace():
                keep.append(i)
        return "".join((txt or "")[i] for i in keep), keep

    def _canon(txt):                       # 숫자 표기를 맞춥니다 — 5.0 → 5 · 3.50 → 3.5 · 1,234 → 1234
        def _one(m):
            s = m.group(0).replace(",", "")
            return s.rstrip("0").rstrip(".") if "." in s else s
        return re.sub(r"\d[\d,]*(?:\.\d+)?", _one, txt or "")

    def _end(line, lab):                   # 라벨이 끝나는 자리 — 정확히 → 공백·괄호를 걷어낸 꼴, 없으면 -1
        if lab and lab in line:
            return line.find(lab) + len(lab)
        tight_label = _tighten(lab)[0]
        if not tight_label:
            return -1
        tight_line, keep = _tighten(line)
        at = tight_line.find(tight_label)
        return (keep[at + len(tight_label) - 1] + 1) if at >= 0 else -1

    tails = []
    for line in val_lines:
        end = _end(line, label)
        if end < 0:                                       # 3단 — 숫자 표기를 맞추고 다시 봅니다
            line = _canon(line)
            end = _end(line, _canon(label))
        if end < 0:
            continue
        tail = line[end:]
        cut_at = [p for p in (tail.find("="), tail.find(":")) if p >= 0]
        tails.append(tail[min(cut_at) + 1:] if cut_at else tail)
    return tails


def _axes(fig):
    """Figure · Axes 어느 쪽을 받아도 축 목록으로 폅니다"""
    if fig is None:
        return []
    axs = getattr(fig, "axes", None)
    if isinstance(axs, list):
        return axs
    return [fig] if hasattr(fig, "get_xlabel") else []


def _plot_gaps(ax, spec):
    """축 하나를 기준과 대조해 어긋난 곳을 문장 목록으로 돌려줍니다 (빈 목록이면 통과).

    그래프 종류(막대·가로 막대·선·점, pandas·seaborn 으로 그린 것)는 가리지 않고
    가로축·세로축 이름만 봅니다.
    """
    gaps = []
    if spec.get("축라벨") and not (ax.get_xlabel().strip() and ax.get_ylabel().strip()):
        gaps.append("가로축·세로축 이름 가운데 빈 것이 있다")
    return gaps


def _one_line(e, n=120):
    """예외를 한 줄로 — 트레이스백 대신 학생이 읽을 문장 하나만 남깁니다"""
    txt = str(e).replace("\n", " ").strip()
    return (txt[:n] + "…") if len(txt) > n else txt


def _prompt1(n=70):
    """지금 셀의 지시 첫 줄 — 튜터에 보낼 질문에 그대로 싣습니다"""
    head_lines = [l.strip() for l in str(_get("PROMPT") or "").split("\n") if l.strip()]
    return (head_lines[0][:n] + "…") if head_lines else "____"


_PROMPT_MIN = 30
_PROMPT_PLACEHOLDERS = ("", "목표", "쓸 변수", "출력", "목표 —", "쓸 변수 —", "출력 —", "TODO",
                    "여기에 내 말로 적습니다")


def _strip(txt):
    """빈칸·줄표·가운뎃점을 떼어 낸 알맹이 — 자리표시자 대조는 이 꼴로 합니다"""
    return re.sub(r"[\s—·:,\-]+", "", str(txt or ""))


_PLACEHOLDERS_STRIPPED = {_strip(t) for t in _PROMPT_PLACEHOLDERS}


def _prompt_missing():
    """지시 입력란이 아직 내 말로 채워지지 않았으면 사유 한 문장, 채워졌으면 None"""
    try:
        txt = str(_ip.user_ns.get("PROMPT", _get("PROMPT")) or "").strip()
    except Exception:
        txt = str(_get("PROMPT") or "").strip()
    stripped = _strip(txt)                          # 길이보다 먼저 본다 — 견본을 그대로 둔 자리가 있다
    if not stripped:
        return "PROMPT 입력란이 비어 있습니다"
    if stripped in _PLACEHOLDERS_STRIPPED:
        return "PROMPT 입력란이 자리표시자·견본 그대로입니다"
    if len(txt) < _PROMPT_MIN:
        return f"PROMPT 입력란이 {len(txt)}자뿐입니다 (최소 {_PROMPT_MIN}자)"
    return None


def _reveal_show(txt):
    """완성본 코드와 해설 — **평문 그대로** 찍는다.

    LMS 렌더러 계약: 「🔓 정답이 열렸습니다 …」 줄부터 채점 표를 닫는 `─` 구분선 줄까지를
    출력 영역이 「🔓 완성본 풀이 보기」 상자로 접는다. 접힘의 표지가 이 **평문 마커**라
    HTML `details` 로 싣지 않는다 — 마커도 이 함수도 바꾸지 않는다.
    """
    print(txt)


_LIVE = {}                                 # 미션마다 이 세션의 가중치로 다시 계산한 기준


def _live_spec(key, spec):
    """MLP 에 걸린 기준값을 이 노트북의 「미션 2 준비 셀」이 정한 가중치로 다시 계산합니다.

    가중치는 준비 셀이 CPU 에서 정하므로, 노트북을 실행한 CPU 에 따라 가중치의 끝자리가 달라지고
    그 차이가 원점 MASE 까지 번집니다. 적어 둔 값에 묶어 두면 바른 코드가 ❌ 를 받으므로,
    바른 입력 윈도(원점 28일 전 ~ 원점 전날)로 이 세션의 모델을 다시 불러 기준의 가운데를 정하고
    허용 폭은 좁게 둡니다(공휴일 없는 두 주 평균 MASE ±0.002). 다시 계산하지 못하면 적어 둔 넓은
    기준(맥 CPU · Colab CPU · T4 에서 잰 값을 모두 담는 범위)을 씁니다. 「원점 전날에서 끝나는
    입력 윈도 수」는 가중치와 무관한 날짜 계산이라 적어 둔 값을 그대로 씁니다.
    """
    lab = "공휴일 없는 두 주 평균 MASE"
    if spec is None or key != "m2" or lab not in spec.get("values", {}):
        return spec
    try:
        import numpy as _np
        import pandas as _pd
        import torch as _torch
        mat = _np.asarray(_get("MAT"), dtype=float)
        dates = _pd.DatetimeIndex(_get("DATES"))
        if mat.ndim != 2 or mat.shape != (100, len(dates)):
            return spec

        def _mase(o, p):                   # 원점 열 o 에서 역 x 7일 예측 p 의 MASE, 분모는 원점 전날까지로
            past = mat[:, :o]
            qden = _np.mean(_np.abs(past[:, 7:] - past[:, :-7]), axis=1)
            return float(_np.mean(_np.abs(mat[:, o:o + 7] - p) / qden[:, None]))

        def _around(ref, center, half, nd):   # 이 세션의 값을 가운데로 두고 허용 폭 half 로 좁힙니다
            c = round(center, nd)
            return (round(c - half, nd), round(c + half, nd)) + tuple(ref[2:])

        models, mu, sd = _get("MODELS"), _get("MU"), _get("SD")
        mase = {}
        for d in ("2025-08-13", "2025-10-01", "2025-12-10", "2026-02-11", "2026-06-24"):
            org = _pd.Timestamp(d)
            o = dates.get_loc(org)
            m_ = _np.asarray(mu[org], dtype=float)[:, None]
            s_ = _np.asarray(sd[org], dtype=float)[:, None]
            dev = next(models[org].parameters()).device
            x = _torch.as_tensor((mat[:, o - 28:o] - m_) / s_, dtype=_torch.float32).to(dev)
            with _torch.no_grad():
                z = models[org](x).cpu().numpy().astype(float)
            mase[d] = _mase(o, z * s_ + m_)
        plain = (mase["2025-12-10"] + mase["2026-06-24"]) / 2
        new = dict(spec)
        vals = dict(spec["values"])
        vals[lab] = _around(spec["values"][lab], plain, 0.002, 3)
        new["values"] = vals
        _LIVE[key] = new
        return new
    except Exception:
        return spec


def _error_hint(key, error):
    """오류에 맞는 (힌트, 물을 것) — 노트북에 있는 이름을 잘못 옮겨 적은 NameError 면 그 이름을 짚습니다.

    그 밖의 오류는 미션 공통 힌트입니다(준비 셀·설치 셀이 만든 이름과 다시 실행할 셀).
    """
    kind, msg = type(error).__name__, str(error)
    if kind == "NameError":                # 노트북에 있는 이름을 잘못 옮겨 적은 경우(FIG_W → FGW)는 그 이름을 짚습니다
        import difflib
        bad = re.search(r"name '(\w+)'", msg)
        near = bad and difflib.get_close_matches(
            bad.group(1), [n for n in globals() if not n.startswith("_") and n != "FIGW"], n=1, cutoff=0.7)
        if near:
            return (f"「{bad.group(1)}」 이름은 이 노트북에 없습니다. 비슷한 이름은 「{near[0]}」 입니다."
                    f" 잘못 옮겨 적었다면 코드에서 「{bad.group(1)}」 를 「{near[0]}」 로 고쳐 다시 실행하세요",
                    "이 오류가 어느 행에서 왜 났는지")
    hint, question = tuple(_HINT.get(key, {}).get("error", ("", "이 오류가 어느 행에서 왜 났는지")))[:2]
    return hint, question


def judge(key, text="", figs=(), error=None):
    """출력 텍스트와 그림을 기준과 대조해 채점 표를 찍습니다 — 통과하면 완성본 풀이를 엽니다"""
    spec = _live_spec(key, _ANS.get(key))   # MLP 에 걸린 기준은 이 세션의 가중치로 다시 계산합니다
    if spec is None:                       # 기준이 아직 없는 미션은 조용히 넘어갑니다
        return None
    head_label = ("미션 " + key[1:]) if key[:1] == "m" and key[1:].isdigit() else key
    failed, passed = [], True
    print("\n" + "─" * 54)
    print(f"◼ {head_label} 채점 결과")
    reason = _prompt_missing()                        # 값·그래프보다 먼저 — 내 말로 시킨 지시가 있는가
    _locked = bool(reason)
    if reason and key != "m2":
        print(f"❌ {reason} — 짧게 시켜도 됩니다. 다만 무엇을 어떤 단위로 출력할지까지 적으면 한 번에 통과할 확률이 높습니다")
        print("   PROMPT 입력란을 채우기 전에는 값·그래프 채점도, 완성본 풀이도 열지 않습니다")
        print('   AI 튜터 대화창이 아니라 이 셀의 PROMPT = """ """ 따옴표 안에 적었는지 확인하세요')
        print("─" * 54)
        return False
    elif reason:                                   # 함정 미션은 PROMPT 가 비어도 표시선 아래 코드를 진단합니다
        print(f"❌ {reason} — 완성본 풀이는 PROMPT 를 채운 뒤에 열립니다. 아래는 AI 코드 표시선 아래 코드의 진단입니다")
        print('   고칠 지시는 AI 튜터 대화창이 아니라 이 셀의 PROMPT = """ """ 따옴표 안에 적습니다')
    if error is not None:                   # 오류가 나면 값·그래프 채점은 뜻이 없습니다
        print(f"❌ 실행 오류 — {type(error).__name__}: {_one_line(error)}")
        hint, question = _error_hint(key, error)          # 비슷한 이름이 있으면 그 이름, 없으면 미션 공통 힌트
        if hint:
            print(f"   힌트 : {hint}")
        form = _ASK.get(key, "full")
        print(_ASK_HEAD[form])
        print("  " + _ASK_FORM[form].format(name="실행 오류", prompt=_prompt1(), ask=question))
        print("─" * 54)
        return False
    txt = text or ""
    if not txt.strip():
        print("❌ 출력이 없습니다 — 값을 print 하도록 프롬프트의 출력 형식에 적으세요")
    units = [str(ref[2]) for ref in spec.get("values", {}).values() if len(ref) > 2]
    val_lines = _value_lines(txt, units)                    # 단위·「=」 가 붙은 행에서 먼저 찾습니다
    all_lines = [l for l in txt.splitlines() if l.strip()]
    for label, ref in spec.get("values", {}).items():
        lo, hi = float(ref[0]), float(ref[1])
        unit = ref[2] if len(ref) > 2 else ""
        want_form = f"「{label} = 숫자{(' ' + unit) if unit else ''}」"
        # 이 이름이 있는 줄의 「=」·「:」 뒤 첫 숫자만 읽습니다. 이름이 있는 줄이 없을 때 출력 전체의 숫자로
        # 물러나면 다른 행의 값(부호를 뒤집은 차, 다른 모델의 MASE)으로 틀린 값이 ✅ 가 됩니다
        tail = _label_lines(val_lines, label)
        picked = _tail_numbers(tail)                    # 꼬리는 괄호를 걷어낸 첫 숫자 하나만 봅니다
        if not picked:                                  # 「이름: 값」 꼴 — 「=」 없는 행에서도 찾습니다
            tail = _label_lines(all_lines, label) or tail
            picked = _tail_numbers(tail)
        hits = [v for v in picked if lo <= v <= hi]
        if hits:
            print(f"✅ {label} — 출력에서 찾은 값: {_num_txt(hits[0])}")
            if unit and unit not in txt:
                print(f"   ⓘ 단위 「{unit}」 도 값과 함께 출력하면 보고서에 그대로 옮길 수 있습니다")
        elif not tail:
            passed = False
            failed.append((label, label))
            print(f"❌ {label} — 이 이름이 있는 줄을 출력에서 찾지 못했습니다. {want_form} 한 줄로 출력하세요")
        else:
            passed = False
            failed.append((label, label))
            _prev_fail = _TRY.get(key, 0)              # 첫 ❌ 에는 기준 범위를 감춥니다
            got_txt = (f"이 이름이 있는 줄에서 읽은 값은 {_num_txt(picked[0])} 인데 기준 범위 밖입니다" if picked
                       else f"이 이름이 있는 줄에서 숫자를 찾지 못했습니다. {want_form} 한 줄로 출력하세요")
            band = (f"기준 {_num_txt(lo, 'g')}" if lo == hi
                    else f"기준 범위 {_num_txt(lo, 'g')} ~ {_num_txt(hi, 'g')}")
            print(f"❌ {label} — {got_txt}" + (f" ({band})" if _prev_fail else ""))
    for phrase in spec.get("forbid", ()):            # 함정 미션 — 전임자의 판단 행이 남으면 통과시키지 않습니다
        if phrase in txt:
            passed = False
            if ("남은 줄", "forbid") not in failed:
                failed.append(("남은 줄", "forbid"))
            print(f"❌ 남은 줄 — 출력에 「{phrase}」 문구가 남아 있습니다")
    plot_spec = spec.get("plot")
    if plot_spec:
        axs = [a for f in figs for a in _axes(f)]
        if not axs:
            passed = False
            failed.append(("그래프", "plot"))
            print("❌ 그래프 — 그림을 찾지 못했습니다 (그래프 한 장도 같은 프롬프트에 함께 시킵니다)")
        else:
            gaps = None
            for ax in axs:                 # 축이 여럿이면 기준에 맞는 축이 하나만 있어도 통과입니다
                try:
                    this_gaps = _plot_gaps(ax, plot_spec)
                except Exception as e:
                    this_gaps = [f"그림의 축 이름을 읽지 못했습니다 ({type(e).__name__})"]
                if not this_gaps:
                    gaps = []
                    break
                if gaps is None or len(this_gaps) < len(gaps):
                    gaps = this_gaps
            if gaps:
                passed = False
                failed.append(("그래프", "plot"))
                print("❌ 그래프 — " + " · ".join(gaps))
            else:                                   # 그래프 종류는 가리지 않고 그림과 축 이름만 봅니다
                print("✅ 그래프 — 그림이 있고 가로축·세로축 이름이 있습니다")
    tbl = _HINT.get(key, {})
    attempt = _TRY[key] = _TRY.get(key, 0) + (1 if failed else 0)
    for name, hkey in failed:
        tiers = tbl.get(hkey, ("",))
        tier = tiers[:-1] if len(tiers) > 2 else tiers[:1]   # 2단 힌트는 마지막 원소가 튜터에 물을 것입니다
        hint = tier[min(attempt, len(tier)) - 1] if tier else ""
        if hint:
            print(f"   힌트({name}) : {hint}")
    if failed:
        form = _ASK.get(key, "full")
        print(_ASK_HEAD[form])
        for name, hkey in failed:
            question = (tbl.get(hkey, ("", ""))[-1] if len(tbl.get(hkey, ("",))) > 1 else "") \
                or "이 항목이 왜 ❌ 로 남는지"
            print("  " + _ASK_FORM[form].format(name=name, prompt=_prompt1(), ask=question))
    elif _locked:                                  # 값·그림은 기준에 맞지만 PROMPT 칸이 아직 비었습니다
        print('ⓘ 값과 그림은 기준에 맞습니다 — 이 셀의 PROMPT = """ """ 따옴표 안에 고칠 지시를 내 말로 쓰고'
              " 다시 실행하면 완성본 풀이가 열립니다")
    else:
        print("🔓 정답이 열렸습니다 — 아래 「🔓 완성본 풀이 보기」를 펼쳐 내 코드와 견줘 보세요")
        _reveal_show(base64.b64decode(_REVEAL[key]).decode("utf-8"))
    print("─" * 54)
    return passed and not _locked


class _Tee:
    """「PROMPT 셀」의 출력을 화면과 버퍼 양쪽으로 흘려보냅니다"""

    def __init__(self, orig, buf):
        self._o, self._b = orig, buf

    def write(self, s):
        self._b.write(s)
        return self._o.write(s)

    def flush(self):
        self._o.flush()

    def __getattr__(self, n):
        return getattr(self._o, n)


def _tag(src):
    """첫 행이 `# 미션 N — 프롬프트` 인 셀만 채점 대상입니다"""
    head_line = (src or "").split("\n")[0]
    m = re.match(r"\s*#\s*미션\s*(\d+)\b", head_line)
    if not m or "프롬프트" not in head_line:                           # 「준비 셀」(`# 미션 N · 준비`)은 대상이 아닙니다
        return None
    return "m" + m.group(1)


def _pre(info=None):
    """「PROMPT 셀」면 출력 버퍼를 걸고 그림 목록을 비웁니다"""
    try:
        tag = _tag(getattr(info, "raw_cell", ""))
        _STATE["tag"] = tag
        if tag is None or tag not in _ANS:
            return
        _FIGS.clear()
        _STATE["ids0"] = {id(f) for f in _open_figs()}
        _STATE["buf"], _STATE["out"] = io.StringIO(), sys.stdout
        sys.stdout = _Tee(sys.stdout, _STATE["buf"])
    except Exception:
        pass


def _post(result=None):
    """「PROMPT 셀」이 끝나면 출력을 되돌리고 그 셀의 출력·그림으로 채점합니다"""
    tag, buf, out = _STATE.get("tag"), _STATE.get("buf"), _STATE.get("out")
    _STATE["tag"], _STATE["buf"], _STATE["out"] = None, None, None
    if out is not None:
        sys.stdout = out
    if tag is None or tag not in _ANS:
        return
    seen = {id(f) for f in _FIGS}
    figs_all = list(_FIGS) + [f for f in _open_figs()
                         if id(f) not in seen and id(f) not in _STATE.get("ids0", set())]
    txt = buf.getvalue() if buf is not None else ""
    last_val = getattr(result, "result", None)       # 셀 마지막 표현식의 값 — print 없이 표로 띄운 값도 읽습니다
    if last_val is not None:
        try:
            txt = txt + "\n" + str(last_val)
        except Exception:
            pass
    try:
        judge(tag, txt, figs_all, error=getattr(result, "error_in_exec", None))
    except Exception as e:
        print(f"(채점을 마치지 못했습니다 — {type(e).__name__}: {_one_line(e)})")


try:
    _ip = get_ipython()
    for _ev, _fn in (("pre_run_cell", _pre), ("post_run_cell", _post)):
        for _old in list(_ip.events.callbacks[_ev]):
            if getattr(_old, "__name__", "") == _fn.__name__:
                _ip.events.unregister(_ev, _old)     # 이 셀을 다시 실행해도 채점은 한 번만 찍힙니다
        _ip.events.register(_ev, _fn)
    print("채점 준비 완료 — 미션의 「PROMPT 셀」을 실행하면 실행 결과(출력)에 채점 결과가 찍힙니다")
except Exception:
    print("채점 준비 완료 (IPython 밖이라 훅 없이 judge() 만 올렸습니다)")

## 서울 지하철 100개 역 준비

자료는 **서울 지하철 일별 승하차** 기록이고, 기간은 2015-01-01～2026-06-30 입니다. 출처는 서울열린데이터광장의 「서울시 지하철 호선별 역별 승하차 인원 정보」(OA-12914)입니다. 계열 하나는 (호선, 역) 한 쌍의 날짜별 승차이고, 이 과목은 계열 하나를 역 하나로 셉니다.

기록이 끊기지 않은 계열 가운데 2024년까지 평균 승차가 많은 100개 역을 고릅니다. 100개 역 × 4,199일 = 419,900개 값에 빈칸이 없는 이 표를 **100개 역 표**라 부릅니다.

「오늘의 준비」 셀은 자료를 처음 받을 때 30초 안팎 걸리고 아래 넷을 출력합니다.

- 표 1 · 베이스라인 셋의 두 묶음 평균 MASE (배)
- 표 2 · 승차 상위 10개 역과 하위 50개 역이 계절 나이브 제곱오차에서 차지하는 비중 (%)
- 표 3 · 예측 원점별 채점 7일과 평일 공휴일 (일)
- 그림 1 · 강남 윈도 한 행, 원점 2026-06-24

<img src="https://skforecast.org/latest/img/backtesting_refit.gif" alt="초록 학습 구간이 오른쪽으로 늘어나고 파란 채점 구간이 그 바로 뒤를 따라 이동하는 애니메이션, 빨간 선이 계열이다" width="640" height="284"/>
<p align="center"><em>▲ 예측 원점을 옮겨 가며 다시 채점하는 방법. 초록 학습 구간이 한 번 옮길 때마다 늘고, 파란 채점 구간이 그 바로 뒤를 따라간다. 파란 구간의 첫날이 예측 원점이다 (출처: skforecast, Backtesting)</em></p>

In [ ]:
# 오늘의 준비 — 읽고 실행만 합니다
# 서울 지하철 100개 역으로 표 1 · 표 2 · 표 3 · 그림 1 을 만듭니다. 자료를 처음 받을 때 30초 안팎 걸립니다
RAW_URL = ("https://github.com/studio-js/seoul-subway-ridership/raw/main/data/"
           "seoul_subway_daily_2015_2026.parquet")
CACHE = "seoul_subway_daily.parquet"          # 한 번 받아 두면 다시 받지 않습니다
if not os.path.exists(CACHE):
    pd.read_parquet(RAW_URL).to_parquet(CACHE)
#  받아지지 않으면: 위 주소의 파일을 내려받아 Colab 왼쪽 파일 목록에 올린 뒤 이 셀을 다시 실행하세요

_raw = pd.read_parquet(CACHE)                  # 한 행이 (날짜, 호선, 역) 하루
_raw["date"] = pd.to_datetime(_raw["date"])
_raw["station"] = _raw["station"].astype(str).str.replace(r"\(.*\)$", "", regex=True)   # 괄호 별칭을 뗀다
_raw["line"] = _raw["line"].astype(str).replace({"9호선2단계": "9호선(2~3단계)",
                                                "9호선2~3단계": "9호선(2~3단계)"})   # 갈린 호선 표기를 하나로
_clean = _raw.groupby(["date", "line", "station"], as_index=False,
                      observed=True)[["board", "alight"]].sum()
_span = _clean.groupby(["line", "station"])["date"].agg(["min", "max", "count"])
_full = _span[(_span["min"] <= pd.Timestamp("2015-01-01")) & (_span["max"] >= pd.Timestamp("2026-06-30"))
              & (_span["count"] >= 4190)]                                    # 기록이 끊기지 않은 계열
_top = (_clean[_clean["date"] <= pd.Timestamp("2024-12-31")].groupby(["line", "station"])["board"]
        .mean().loc[_full.index].sort_values(ascending=False).index[:100])   # 2024년까지 평균 승차 상위 100개 역
_wide = _clean.pivot_table(index=["line", "station"], columns="date", values="board").loc[_top]
MAT = _wide.to_numpy(dtype=float)              # 100개 역 x 4,199일 승차 (명)
DATES = pd.DatetimeIndex(_wide.columns)        # 열의 날짜
STATIONS = [f"{_a} {_b}" for _a, _b in _wide.index]   # 행의 역 이름 (0번 = 2호선 강남)
ORIGINS = pd.to_datetime(["2025-08-13", "2025-10-01", "2025-12-10",
                          "2026-02-11", "2026-06-24"])   # 예측 원점 5개(모두 수요일)
PLAIN_ORIGINS = ORIGINS[[2, 4]]                # 공휴일 없는 두 주
HOL_ORIGINS = ORIGINS[[0, 1, 3]]               # 공휴일 있는 세 주
H = 7                                          # 채점 7일(원점 당일부터)
print(f"100개 역 표 : {MAT.shape[0]}개 역 x {MAT.shape[1]:,}일 ({DATES[0].date()} ~ {DATES[-1].date()})"
      f" · 빈칸 {int(np.isnan(MAT).sum())}개")


def _mase(o, pred):
    """원점 열 o 에서 역 x 7일 예측(명)의 역 단위 MASE. 분모는 원점 전날까지 계절 나이브의 평균 절대 오차"""
    past = MAT[:, :o]
    qden = np.mean(np.abs(past[:, 7:] - past[:, :-7]), axis=1)
    return float(np.mean(np.abs(MAT[:, o:o + H] - pred) / qden[:, None]))


# 표 1: 원점마다 베이스라인 셋을 역 단위 MASE 로 채점하고 두 묶음으로 나눠 평균한다
_by = {"naive": [], "snaive": [], "ma7": []}
for _og in ORIGINS:
    _o = DATES.get_loc(_og)
    _by["naive"].append(_mase(_o, np.repeat(MAT[:, _o - 1:_o], H, axis=1)))        # 원점 전날 값 하나를 7일에
    _by["snaive"].append(_mase(_o, MAT[:, _o - 7:_o]))                              # 1주일 전 같은 요일 값
    _by["ma7"].append(_mase(_o, np.repeat(MAT[:, _o - 7:_o].mean(axis=1, keepdims=True), H, axis=1)))  # 직전 7일 평균
BASE = {k: (float(np.mean([v[2], v[4]])), float(np.mean([v[0], v[1], v[3]])), float(np.mean(v)))
        for k, v in _by.items()}               # (공휴일 없는 두 주, 공휴일 있는 세 주, 전체) 평균 MASE
_names = {"naive": "나이브 기법", "snaive": "계절 나이브", "ma7": "직전 7일 평균"}
print("표 1 · 베이스라인 셋의 두 묶음 평균 MASE (배)")
for _k in ("naive", "snaive", "ma7"):
    print(f"  {_names[_k]} : 공휴일 없는 두 주 {BASE[_k][0]:.3f} · 공휴일 있는 세 주 {BASE[_k][1]:.3f}"
          f" · 전체 평균 {BASE[_k][2]:.3f}")

# 표 2: 원점 2026-06-24 의 학습 구간(원점 57일 전까지)에서 계절 나이브 제곱오차를 역마다 더해 묶음별 비중을 잰다
_o = DATES.get_loc(pd.Timestamp("2026-06-24"))
_cut = _o - 56                                  # 원점 직전 56일(검증 구간)은 뺀다
_hist = MAT[:, :_o]                             # 원점 전날까지
_rank = np.argsort(-_hist[:, -365:].mean(axis=1))   # 승차 순위 = 원점 전 365일 평균 승차
_mu = _hist[:, :_cut].mean(axis=1, keepdims=True)   # 역마다 원점 57일 전까지로 잰 평균 · 표준편차
_sd = _hist[:, :_cut].std(axis=1, keepdims=True)
_z = (_hist - _mu) / _sd                          # 역마다 표준화한 승차
_se = {"raw": ((_hist[:, 7:_cut] - _hist[:, :_cut - 7]) ** 2).sum(axis=1),   # 명 단위 제곱오차
       "std": ((_z[:, 7:_cut] - _z[:, :_cut - 7]) ** 2).sum(axis=1)}         # 표준화한 값의 제곱오차
SHARE = {f"{_k}_{_g}": float(_se[_k][_idx].sum() / _se[_k].sum())
         for _k in ("raw", "std") for _g, _idx in (("top10", _rank[:10]), ("bottom50", _rank[50:]))}
print("표 2 · 승차 상위 10개 역과 하위 50개 역이 계절 나이브 제곱오차에서 차지하는 비중 (%)")
for _g, _n in (("top10", "승차 상위 10개 역 (역 수 10%)"), ("bottom50", "승차 하위 50개 역 (역 수 50%)")):
    print(f"  {_n} : 표준화 없이 {100 * SHARE['raw_' + _g]:.1f} · 역마다 표준화 {100 * SHARE['std_' + _g]:.1f}")

# 표 3: 채점 7일 안의 평일(월~금) 공휴일. 대한민국 공휴일 가운데 원점 5개의 채점 7일 안에 있는 날만 옮겼다
KR_HOLIDAYS = {"2025-08-15": "광복절", "2025-10-03": "개천절", "2025-10-05": "추석 전날",
               "2025-10-06": "추석", "2025-10-07": "추석 다음 날", "2026-02-16": "설 전날",
               "2026-02-17": "설날"}
WH_COUNT = {}                                   # 원점 날짜 -> 채점 7일 가운데 평일 공휴일 수
print("표 3 · 예측 원점별 채점 7일과 평일 공휴일 (일)")
for _og in ORIGINS:
    _days = pd.date_range(_og, periods=H)
    _wh = [d for d in _days if d.strftime("%Y-%m-%d") in KR_HOLIDAYS and d.dayofweek < 5]
    WH_COUNT[str(_og.date())] = len(_wh)
    _txt = " · ".join(f"{d:%m-%d} {KR_HOLIDAYS[d.strftime('%Y-%m-%d')]}" for d in _wh) or "없음"
    print(f"  원점 {_og.date()}   채점 {_days[0]:%m-%d} ~ {_days[-1]:%m-%d}   평일 공휴일 {len(_wh)}일   {_txt}")

# 그림 1: 원점 2026-06-24 에서 강남의 윈도 한 행. 입력 28일은 원점 전날에서 끝나고 출력 7일은 원점 당일부터
_g, _o = STATIONS.index("2호선 강남"), DATES.get_loc(pd.Timestamp("2026-06-24"))
fig, ax = plt.subplots(figsize=(FIG_W, 3.6))
ax.plot(DATES[_o - 28:_o], MAT[_g, _o - 28:_o] / 1000, marker="o", ms=3, color=COL["파랑"],
        label="입력 28일 (05-27 ~ 06-23)")
ax.plot(DATES[_o:_o + H], MAT[_g, _o:_o + H] / 1000, "o", color=COL["남색"], label="출력 7일 (06-24 ~ 06-30)")
ax.axvline(DATES[_o], color=COL["회색"], ls="--", lw=1, label="예측 원점")
ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=2))   # 눈금은 수요일마다(원점과 같은 요일)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
ax.set_title("그림 1 · 강남 윈도 한 행, 원점 2026-06-24")
ax.set_ylabel("하루 승차 (천 명)")
ax.set_ylim(0, 130)
ax.legend(frameon=False, loc="upper left", ncol=3)
plt.show()

## 손실과 계열별 표준화 확인

> **[문제] 채점 지표가 다른 두 팀은 각각 어떤 손실로 학습하나**
>
> 예를 들어 A 팀은 역 단위 MASE 로, B 팀은 RMSE 로 다음 주 예측을 채점합니다.
>
> 1. 두 팀 모두 MSE: 신경망은 제곱 오차로 학습하는 것이 기본이다
> 2. A 팀은 MAE, B 팀은 MSE: 손실을 각자 채점 지표에 맞춘다
> 3. 두 팀 모두 MAE: 드문 날 하나가 예측을 덜 바꿔 어느 지표에서나 낫다

> **[문제] 표준화한 예측을 채점하기 전에 무엇을 하나**
>
> 예를 들어 큰 역(평균 80,000명, 표준편차 20,000명)과 작은 역(평균 16,000명, 표준편차 4,000명)에서 모델이 두 역 모두 표준화 예측 0.5 를 냈습니다.
>
> 1. 두 역 모두 0.5 를 그대로 채점에 쓴다: 표준화한 값끼리 비교하면 된다
> 2. 100개 역 전체의 평균·표준편차 하나로 두 역을 함께 명으로 되돌린다
> 3. 역마다 자기 표준편차를 곱하고 자기 평균을 더해 명으로 되돌린다

## 서울 지하철 100개 역으로 판단하기

아래 세 문항은 「오늘의 준비」 셀 출력을 읽고 답합니다. 표 1 로 새 신경망을 견줄 베이스라인을 정하고, 표 2 로 표준화 없이 모은 손실을 믿어도 되는지 보고, 표 3 으로 원점 5개를 두 묶음으로 나눕니다.

> **[문제] 새 신경망을 견줄 베이스라인**
>
> 「오늘의 준비」 셀을 실행하면 나오는 「표 1 · 베이스라인 셋의 두 묶음 평균 MASE (배)」를 봅니다. 새 신경망의 MASE 는 어느 베이스라인과 견줘 투입을 판단할까요?
>
> 1. 계절 나이브: 두 묶음 모두 셋 가운데 가장 낮다
> 2. 직전 7일 평균: 지금 인력을 정하는 방식이라 이것보다 MASE 가 낮으면 된다
> 3. MASE 1: 1 보다 작으면 같은 주의 계절 나이브보다 덜 빗나간 것이다

> **[문제] 표준화 없이 모은 모델의 작은 손실**
>
> 「오늘의 준비」 셀을 실행하면 나오는 「표 2 · 승차 상위 10개 역과 하위 50개 역이 계절 나이브 제곱오차에서 차지하는 비중 (%)」를 봅니다. 역마다 오차 크기는 승차 규모를 따르므로, 1주일 전 값을 옮겨 적는 계절 나이브의 제곱오차 비중으로 모델 손실에서 각 역이 차지할 무게를 어림합니다. 표준화 없이 100개 역을 모아 제곱 오차로 학습한 모델의 손실이 작으면, 100개 역을 고르게 맞힌다고 믿어도 될까요?
>
> 1. 믿어도 된다: 100개 역이 같은 무게로 손실에 들어간다
> 2. 믿기 어렵다: 계절 나이브 제곱오차의 약 3분의 1 이 승차 상위 10개 역에서 나온다
> 3. 믿기 어렵다: 계절 나이브 제곱오차의 대부분이 승차 하위 50개 역에서 나온다

> **[문제] 공휴일 있는 세 주의 원점 고르기**
>
> 「오늘의 준비」 셀을 실행하면 나오는 「표 3 · 예측 원점별 채점 7일과 평일 공휴일 (일)」을 봅니다. 채점 7일에 평일 공휴일이 있어 「공휴일 있는 세 주」로 묶이는 원점은?
>
> 1. 2025-08-13 · 2025-10-01 · 2026-02-11
> 2. 2025-10-01 · 2025-12-10 · 2026-02-11
> 3. 2025-08-13 · 2025-10-01 · 2026-06-24

## 미션 1 — MLP(MSE 손실)과 MLP(MAE 손실)의 MASE 차

표 1 에서 새 신경망은 계절 나이브와 견준다고 정했습니다. 손실만 다른 글로벌 MLP 두 개를 같은 원점에서 학습해 그 기준으로 읽습니다.

#### 할 일 — 미션 1 (25분)

1. 아래 「미션 1 준비 셀」을 실행합니다. 이 셀은 고치지 않습니다.
2. 아래 「만들 것」대로 시킬 일을 내 말로 적습니다. AI 튜터 대화창이 아니라 「PROMPT 셀」의 `PROMPT = """ """` 따옴표 안에 적어야 채점되고, 예시 프롬프트도 그 안에 붙여 넣습니다.
3. 셀을 클릭한 채 AI 튜터에게 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶ 합니다. LMS 밖이면 받은 코드를 표시선 아래에 둡니다.
4. 채점 결과를 봅니다. ❌ 면 힌트대로 PROMPT 를 고쳐 3번부터 다시 합니다.

**이 미션의 질문**: 손실만 MSE 에서 MAE 로 바꾸면 공휴일 없는 주와 공휴일 있는 주의 MASE 는 각각 커질까요, 작아질까요?

**쓸 수 있는 것**: 「미션 1 준비 셀」이 만들어 둔 이름입니다.

| 이름 | 무엇 |
|---|---|
| `ORIGIN_PLAIN` · `ORIGIN_HOLIDAY` | 원점 "2025-12-10"(공휴일 없는 주) · 원점 "2025-10-01"(공휴일 있는 주) |
| `fit_global(origin, loss, seed=SEED)` | 원점 하나에서 100개 역 글로벌 MLP 를 학습하고, 채점 7일의 MASE(`mase`)와 조기 종료가 되돌린 에폭(`best_epoch`)을 담은 사전을 낸다. `loss` 는 "mse" 또는 "mae" |
| `SNAIVE_TWO` | 두 원점의 계절 나이브 MASE, 키는 원점 날짜 문자열 |
| `SEED` · `DEVICE` | 시드 0 · 학습과 예측을 실행할 곳 |
| `COL` · `FIG_W` | 그래프 색 이름 사전(`COL["남색"]` 등)과 그림 가로 크기(설치 셀) |

#### 만들 것

1. **공휴일 없는 주 MASE 차(MSE − MAE)**: 단위 없음 · 소수 셋째 자리
   - `ORIGIN_PLAIN` 에서 MLP(MSE 손실) MASE 에서 MLP(MAE 손실) MASE 를 뺀 값입니다. 양수면 MLP(MAE 손실)이 덜 빗나갔습니다
2. **공휴일 있는 주 MASE 차(MSE − MAE)**: 단위 없음 · 소수 셋째 자리
   - `ORIGIN_HOLIDAY` 에서 같은 순서로 뺀 값입니다
3. **그래프 한 장**: 종류는 자유이고 가로축·세로축에 이름을 답니다. 권장은 그림 2개를 나란히 둔 막대 그래프입니다
   - 왼쪽 그림은 공휴일 없는 주, 오른쪽 그림은 공휴일 있는 주이고 그림 제목에 원점 날짜를 적습니다
   - 그림마다 계절 나이브 · MLP(MSE 손실) · MLP(MAE 손실) 막대 3개, 가로축 이름 「모델」, 세로축 이름 「MASE」

**통과 기준**: 값 2개가 「이름 = 값」 한 행씩 있고, 가로축·세로축 이름이 있는 그래프가 있으면 통과입니다. 채점은 기기마다 조금씩 다른 MASE 를 감안해 두 차의 부호와 대략의 크기를 봅니다. 학습 네 번에 CPU 로 2～3분 걸리고, PROMPT 가 비어 있으면 학습이 끝난 뒤에야 잠김 안내가 나오니 PROMPT 부터 채웁니다.

#### 프롬프트에 넣을 것(힌트)

- **고정할 이름**: 위 표의 이름은 이미 있으니 다시 만들지 말라고 적습니다.
- **두 모델**: `fit_global` 을 원점 둘 × 손실 둘로 네 번 부르고, 시드와 DEVICE 는 같게 둔다고 적습니다.
- **빼는 순서와 출력**: 값 두 개의 이름, MLP(MSE 손실) − MLP(MAE 손실) 순서, 소수 셋째 자리를 적습니다.
- **그래프**: 위 3번의 권장 그래프와 가로축·세로축 이름을 적습니다.

<details class="lms-extra"><summary>자세히 — 예시 프롬프트</summary>

위 표의 이름은 새로 만들지 말고, 두 원점마다 fit_global 을 loss="mse" 와 loss="mae" 로 한 번씩 불러 원점과 손실마다 MASE 를 한 행씩 찍어 줘. 두 차는 MLP(MSE 손실)에서 MLP(MAE 손실)을 뺀 값을 소수 셋째 자리까지 "공휴일 없는 주 MASE 차(MSE − MAE) = 값" 처럼 출력해 줘. 공휴일 있는 주도 "공휴일 있는 주 MASE 차(MSE − MAE) = 값" 으로 출력해 줘. 그림 두 개를 나란히 두고 그림마다 계절 나이브 · MLP(MSE 손실) · MLP(MAE 손실) 막대 세 개와 원점 날짜 제목, 가로축 「모델」, 세로축 「MASE」 를 달아 줘.

</details>

<details class="lms-extra"><summary>배경 — MSE 와 MAE 가 고르는 예측값</summary>

| 항목 | MLP(MSE 손실) | MLP(MAE 손실) |
|---|---|---|
| 손실 | 오차를 제곱해 평균 | 오차의 절댓값을 평균 |
| 여러 값을 한 값으로 예측할 때 손실이 가장 작은 값 | 평균 | 중앙값 |
| 평일 공휴일 목표값(전체 목표값의 3.8%)이 손실 합에서 차지하는 비중 | 59.7% | 32.6% |

오차를 제곱하면 크게 빗나간 날 하나가 손실에서 훨씬 큰 비중을 차지합니다. 마지막 행은 원점 2026-06-24 의 100개 역 학습 윈도에서 학습을 마친 모델로 계산한 값입니다. 평일 공휴일 목표값은 평소보다 훨씬 작아서, 아래 그림과 반대로 평균이 중앙값보다 작습니다.

<img src="https://commons.wikimedia.org/wiki/Special:FilePath/Visualisation_mode_median_mean.svg" alt="오른쪽으로 긴 꼬리가 있는 같은 분포를 세 번 그리고, 위부터 최빈값 · 중앙값 · 평균의 위치를 세로선과 받침점으로 나타낸 그림" width="320" height="549"/>
<p align="center"><em>▲ 오른쪽 꼬리가 긴 분포의 최빈값 · 중앙값 · 평균. 초록 세로선이 넓이를 50% 씩 나누는 중앙값, 파란 받침점이 평균이고, 평균은 중앙값보다 꼬리 쪽인 오른쪽에 있다. MLP(MSE 손실)의 예측은 평균 쪽, MLP(MAE 손실)의 예측은 중앙값 쪽이다 (출처: Wikimedia Commons)</em></p>

</details>

#### 예상 적기

> **[예측] 손실만 MAE 로 바꾸면 두 원점의 MASE 는 MLP(MSE 손실)에 견줘 어떻게 바뀔까요?**
>
> 1. 두 원점 모두 낮아진다
> 2. 공휴일 없는 주는 낮아지고 공휴일 있는 주는 높아진다
> 3. 두 원점 모두 거의 그대로다
>
> 답을 정한 뒤 아래를 읽어 보세요(정답 채점은 없습니다).

In [ ]:
# 미션 1 · 준비 — 읽고 실행만 합니다. 이 셀이 미션 1 에 쓸 이름을 직접 만듭니다
RAW_URL = ("https://github.com/studio-js/seoul-subway-ridership/raw/main/data/"
           "seoul_subway_daily_2015_2026.parquet")
CACHE = "seoul_subway_daily.parquet"          # 한 번 받아 두면 다시 받지 않습니다
if not os.path.exists(CACHE):
    pd.read_parquet(RAW_URL).to_parquet(CACHE)

# 「오늘의 준비」와 같은 손질: 역 이름의 괄호 별칭을 떼고 9호선 이름을 맞춘 뒤 (날짜, 호선, 역) 으로 합칩니다
_raw = pd.read_parquet(CACHE)
_raw["date"] = pd.to_datetime(_raw["date"])
_raw["station"] = _raw["station"].astype(str).str.replace(r"\(.*\)$", "", regex=True)
_raw["line"] = _raw["line"].astype(str).replace({"9호선2단계": "9호선(2~3단계)",
                                                "9호선2~3단계": "9호선(2~3단계)"})
_clean = _raw.groupby(["date", "line", "station"], as_index=False,
                      observed=True)[["board", "alight"]].sum()
_span = _clean.groupby(["line", "station"])["date"].agg(["min", "max", "count"])
_full = _span[(_span["min"] <= pd.Timestamp("2015-01-01")) &            # 기록이 끊기지 않은 계열만 남긴 뒤
              (_span["max"] >= pd.Timestamp("2026-06-30")) & (_span["count"] >= 4190)]
_top = (_clean[_clean["date"] <= pd.Timestamp("2024-12-31")]             # 2024년까지 평균 승차 상위 100개 역
        .groupby(["line", "station"])["board"].mean()
        .loc[_full.index].sort_values(ascending=False).index[:100])
_wide = _clean.pivot_table(index=["line", "station"], columns="date", values="board").loc[_top]

MAT = _wide.to_numpy(dtype=float)             # 100개 역 x 4,199일 승차 (명)
DATES = pd.DatetimeIndex(_wide.columns)       # 열의 날짜
STATIONS = [f"{_line} {_name}" for _line, _name in _wide.index]   # 행의 역 이름

L, H, VAL_DAYS = 28, 7, 56                    # 입력 28일 · 출력 7일 · 검증 구간 56일
ORIGIN_PLAIN = "2025-12-10"                  # 공휴일 없는 주의 원점 (채점 7일에 평일 공휴일 없음)
ORIGIN_HOLIDAY = "2025-10-01"                # 공휴일 있는 주의 원점 (채점 7일에 개천절 · 추석 · 추석 다음 날)
SEED = 0
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"   # 설치 셀과 같은 규칙, 이 셀만 실행해도 정해집니다


def origin_ctx(origin):
    """원점 하나의 자료: 원점 전날까지의 100개 역(hist), 채점 7일(test, 명), 역별 MASE 분모(qden), 원점의 열 번호(o)"""
    o = int(DATES.get_loc(pd.Timestamp(origin)))
    hist = MAT[:, :o]
    qden = np.nanmean(np.abs(hist[:, 7:] - hist[:, :-7]), axis=1)   # 원점 전날까지로 계산한 분모
    return dict(o=o, hist=hist, test=MAT[:, o:o + H], qden=qden)


def zstats(A):
    """역별 평균과 표준편차 (표준편차가 0 인 역은 1 로 둡니다)"""
    mu = np.nanmean(A, axis=1, keepdims=True)
    sd = np.nanstd(A, axis=1, keepdims=True)
    sd[sd == 0] = 1.0
    return mu, sd


def make_windows(M, t_lo, t_hi):
    """첫 목표일의 열 번호가 t_lo ~ t_hi 인 (입력 28일, 정답 7일) 윈도를 역마다 모두 뽑아 한 표로 모읍니다"""
    W = np.lib.stride_tricks.sliding_window_view(M, L + H, axis=1)
    Wsel = W[:, t_lo - L:t_hi - L + 1, :]
    X = Wsel[..., :L].reshape(-1, L).astype(np.float32)
    Y = Wsel[..., L:].reshape(-1, H).astype(np.float32)
    ok = np.isfinite(X).all(1) & np.isfinite(Y).all(1)
    return X[ok], Y[ok]


class MLP(nn.Module):
    """입력 28 → 64 → ReLU → 64 → ReLU → 출력 7. seed 가 첫 가중치를 정합니다"""

    def __init__(self, seed=0, hidden=64):
        torch.manual_seed(seed)
        super().__init__()
        self.net = nn.Sequential(nn.Linear(L, hidden), nn.ReLU(),
                                 nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, H))

    def forward(self, x):
        return self.net(x)


def train_loop(model, Xtr, Ytr, Xva, Yva, loss="mse", batch=256, max_epochs=30, patience=5,
               seed=0, device=DEVICE):
    """조기 종료를 붙인 학습: 검증 손실이 patience 에폭 이어서 나아지지 않으면 멈추고 가장 좋았던 에폭으로 되돌립니다"""
    np.random.seed(seed)
    torch.manual_seed(seed)
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
    lossf = nn.MSELoss() if loss == "mse" else nn.L1Loss()          # 손실은 이 한 줄에서만 갈린다
    Xtr_t, Ytr_t = torch.from_numpy(Xtr).to(device), torch.from_numpy(Ytr).to(device)
    Xva_t, Yva_t = torch.from_numpy(Xva).to(device), torch.from_numpy(Yva).to(device)
    gen = torch.Generator().manual_seed(seed)          # 섞는 순서는 CPU 에서 정해 DEVICE 가 달라도 같게 둡니다
    n = len(Xtr_t)
    best, best_state, best_ep, bad, stop_ep = float("inf"), None, -1, 0, None
    for ep in range(1, max_epochs + 1):
        model.train()
        perm = torch.randperm(n, generator=gen)
        for i in range(0, n, batch):
            idx = perm[i:i + batch].to(device)
            optimizer.zero_grad()
            lossf(model(Xtr_t[idx]), Ytr_t[idx]).backward()
            optimizer.step()                           # 경사하강법 한 걸음: 손실이 줄어드는 쪽으로 가중치를 옮긴다
        model.eval()
        with torch.no_grad():
            val = lossf(model(Xva_t), Yva_t).item()    # 검증 구간의 윈도에서 잰 손실
        if val < best - 1e-7:                          # 지금까지 가장 낮으면 가중치를 보관한다
            best, best_ep, bad = val, ep, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:                        # patience 에폭 이어서 나아지지 않으면 멈춘다
                stop_ep = ep
                break
    model.load_state_dict(best_state)                  # 멈춘 에폭이 아니라 가장 좋았던 에폭의 가중치로
    model.eval()
    return dict(stop_epoch=stop_ep, best_epoch=best_ep, best_val=best)


def predict(model, x, mu, sd):
    """입력의 끝 28일로 7일을 예측하고 표준화를 풀어 명 단위로 바꿉니다 (모델과 같은 DEVICE 에서 계산)"""
    dev = next(model.parameters()).device
    with torch.no_grad():
        out = model(torch.from_numpy(np.asarray(x)[:, -L:].astype(np.float32)).to(dev))
    return out.cpu().numpy().astype(float) * sd + mu


def score(y, pred, qden):
    """MASE(역마다 분모로 나눈 절대 오차의 평균)와 RMSE(명)를 사전으로 돌려줍니다"""
    y, pred = np.asarray(y, dtype=float), np.asarray(pred, dtype=float)
    q = np.asarray(qden, dtype=float).reshape(-1, 1)
    return dict(mase=float(np.mean(np.abs(y - pred) / q)),
                rmse=float(np.sqrt(np.mean((y - pred) ** 2))))


def fit_global(origin, loss="mse", seed=SEED, device=DEVICE):
    """원점 전날에서 끝나는 입력 28일로 100개 역 글로벌 MLP 를 학습하고 채점 7일 MASE 를 돌려준다.

    loss : "mse" · "mae"
    반환 : mase · point(100 x 7 예측, 명) · y(채점 7일 실제, 명) · qden · best_epoch · stop_epoch · device
    """
    if loss not in ("mse", "mae"):
        raise ValueError(f'loss 는 "mse" 또는 "mae" 입니다 (지금 {loss!r})')
    ctx = origin_ctx(origin)
    hist, o = ctx["hist"], ctx["o"]
    val_first = o - VAL_DAYS                            # 원점 직전 56일의 첫날
    mu, sd = zstats(hist[:, :val_first])                # 계열별 표준화 통계 = 원점 57일 전까지
    M = (hist - mu) / sd
    Xtr, Ytr = make_windows(M, L, val_first - H)        # 학습 윈도: 목표 7일이 원점 57일 전까지 끝난다
    Xva, Yva = make_windows(M, val_first, o - H)        # 검증 구간의 윈도: 목표 7일이 원점 직전 56일 안에서 끝난다
    model = MLP(seed=seed)
    res = train_loop(model, Xtr, Ytr, Xva, Yva, loss=loss, seed=seed, device=device)   # 손실만 인자로 바뀝니다
    pred = predict(model, M, mu, sd)                    # 원점 전날에서 끝나는 28일 → 7일, 명 단위
    mase = score(ctx["test"], pred, ctx["qden"])["mase"]  # 채점 7일은 여기에서만 씁니다
    return dict(origin=origin, loss=loss, mase=mase, point=pred, y=ctx["test"], qden=ctx["qden"],
                best_epoch=res["best_epoch"], stop_epoch=res["stop_epoch"],
                device=str(next(model.parameters()).device))


SNAIVE_TWO = {}                               # 두 원점의 계절 나이브 MASE
for _og in (ORIGIN_PLAIN, ORIGIN_HOLIDAY):
    _c = origin_ctx(_og)
    SNAIVE_TWO[_og] = score(_c["test"], _c["hist"][:, -7:], _c["qden"])["mase"]   # 원점 앞 7일을 다음 7일로

# 본문 수 확인: 강남역, 개천절이 있는 주의 평일 5일 (자료가 바뀌면 여기서 멈춥니다)
_gn = STATIONS.index("2호선 강남")
_wk = MAT[_gn, DATES.get_loc(pd.Timestamp("2025-09-29")):DATES.get_loc(pd.Timestamp("2025-10-03")) + 1]
assert [int(v) for v in _wk] == [88403, 90415, 89889, 89738, 43721]
assert (round(SNAIVE_TWO[ORIGIN_PLAIN], 3), round(SNAIVE_TWO[ORIGIN_HOLIDAY], 3)) == (0.419, 4.388)
print(f"100개 역 표 : {MAT.shape[0]}개 역 x {MAT.shape[1]:,}일 · {DATES[0].date()} ~ {DATES[-1].date()}")
print(f"원점 {ORIGIN_PLAIN}(공휴일 없는 주) · 원점 {ORIGIN_HOLIDAY}(공휴일 있는 주) : 원점마다 원점 전날까지의 자료로 다시 학습합니다")
print("계절 나이브 MASE :", " · ".join(f"{k} {v:.3f}" for k, v in SNAIVE_TWO.items()))
print(f"학습 조건 : 입력 {L}일 · 출력 {H}일 · 검증 구간 원점 직전 {VAL_DAYS}일 · 조기 종료 5에폭"
      f" · 시드 {SEED} · DEVICE {DEVICE}. 손실만 fit_global 의 loss 로 고릅니다")
print("쓸 수 있는 이름 : ORIGIN_PLAIN · ORIGIN_HOLIDAY · fit_global(origin, loss, seed=SEED)"
      " · SNAIVE_TWO · SEED · DEVICE")

In [ ]:
# 미션 1 — 프롬프트   # ai: prompt
# [heavy] 학습을 네 번 해서 CPU 로 2～3분 걸립니다. PROMPT 를 비우면 학습이 끝난 뒤에야 ❌ 가 나옵니다
# PROMPT 를 쓰고 → AI 튜터에 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶
# 시킬 일은 AI 튜터 대화창이 아니라 아래 따옴표 안에 적습니다. 예시 프롬프트도 따옴표 안에 붙여 넣어야 채점됩니다
PROMPT = """

"""
# --- 여기부터 AI 코드 ---

## 미션 2 — 입력 윈도 누수 수정

전임 담당자는 글로벌 MLP(MSE 손실)의 평균 MASE 가 표 1 의 계절 나이브보다 크다며 신경망을 버렸습니다. 그 코드는 입력 28일을 자르는 코드 행이 틀렸습니다. 미션 1 과 같은 순서로 합니다. 다만 먼저 「PROMPT 셀」을 그대로 ▶ 해 전임자 코드의 채점 결과를 봅니다.

**이 미션의 질문**: 입력 윈도를 원점 전날에서 끝나게 고치면 공휴일 없는 두 주의 평균 MASE 는 얼마일까요?

**쓸 수 있는 것**: 「미션 2 준비 셀」이 만들어 둔 이름입니다.

| 이름 | 무엇 |
|---|---|
| `MAT` · `DATES` | 100개 역 표(100개 역 × 4,199일 승차, 명) · 열의 날짜 |
| `ORIGINS` · `PLAIN_ORIGINS` · `HOL_ORIGINS` · `L` · `H` | 원점 5개 · 공휴일 없는 두 주 · 공휴일 있는 세 주 · 입력 28일 · 채점 7일 |
| `qden_of(origin)` · `score(y, pred, qden)` | 원점 전날까지로 잰 역별 MASE 분모 · MASE 와 RMSE(명)를 사전으로 내는 채점 함수 |
| `mlp_forecast(origin, raw28)` | 역 × 28일 승차(명)를 받아 그 원점 MLP 의 역 × 7일 예측(명)을 낸다 |

#### 고칠 것

1. **원점 전날에서 끝나는 입력 윈도 수**: 단위 없음 · 정수
   - 원점마다 입력 윈도의 첫날 ～ 끝날, 끝날에서 원점까지 며칠인지, MASE 를 한 행에 출력합니다. 원점 2025-12-10 이면 「입력 윈도 2025-11-12 ～ 2025-12-09 | 끝날에서 원점까지 1일」입니다. 이렇게 1일인 원점을 셉니다
   - 날짜를 찍는 행(`first, last = …`)도 자른 범위와 같은 번호로 고칩니다
2. **공휴일 없는 두 주 평균 MASE**: 단위 없음 · 소수 셋째 자리
   - 공휴일 없는 두 주인 `PLAIN_ORIGINS` 두 원점의 MASE 를 평균합니다
3. **그래프 한 장**: 종류는 자유이고 가로축·세로축에 이름을 답니다. 권장은 원점 5개의 MASE 막대 5개, 가로축 이름 「원점」, 세로축 이름 「MASE」입니다
4. **판단 문장 지우기**: 「크게 빗나갑니다」 행과 「판단 : 신경망은 버립니다」 행은 점수 하나로 단정한 문장이라 뺍니다. 앞 행 대신 공휴일 있는 세 주 평균과 원점 5개 평균 MASE 만 찍습니다

**통과 기준**: 1·2번 값이 「이름 = 값」 한 행씩 있고, 4번의 두 문장이 없고, 가로축·세로축 이름이 있는 그래프가 있으면 통과입니다. 원점 5개를 모두 고쳐야 1·2번 값이 기준에 맞습니다.

#### 프롬프트에 넣을 것(힌트)

- **고칠 곳**: 입력 28일의 첫날과 끝날을 어느 날로 둘지, 자르는 행과 날짜를 찍는 행을 함께 고치라고 적습니다.
- **그대로 둘 것**: 위 표의 함수는 다시 만들지 말고 `mlp_forecast` 로 예측하라고 적습니다.
- **출력**: 원점마다 날짜 행, 값 두 개의 이름과 자릿수, 판단 문장 두 행 지우기를 적습니다.

<details class="lms-extra"><summary>배경 — 원점의 입력 28일</summary>

이론의 윈도 한 행에서 출력 첫날은 입력 끝날의 다음 날입니다. 원점 당일이 출력 첫날이므로 원점 2025-12-10 의 입력 28일은 2025-11-12 ～ 2025-12-09 입니다.

`mlp_forecast` 의 가중치도 입력 끝날 다음 날부터 7일을 내도록 학습됐습니다. 입력이 원점 당일에서 끝나면 예측은 원점 다음 날부터의 7일이 되어, 채점 7일과 요일이 하루씩 어긋납니다. 그래서 이 누수는 점수를 오히려 나쁘게 만들고, 누수인지는 점수가 아니라 입력 윈도의 날짜로 확인합니다.

<img src="https://otexts.com/fpp3/fpp_files/figure-html/cv1-1.png" alt="행마다 파란 점 뒤에 주황 점 하나와 회색 점이 이어지고, 아래 행일수록 파란 점이 하나씩 늘어나는 그림" width="640" height="200"/>
<p align="center"><em>▲ 시계열 교차 검증(time series cross-validation). 한 행이 한 분할이고 파란 점은 예측에 쓴 관측, 주황 점은 채점하는 시각이다. 두 점이 겹치지 않듯 입력 끝날과 채점 첫날도 다른 날이다 (출처: Hyndman &amp; Athanasopoulos, Forecasting: Principles and Practice 3rd ed., §5.10)</em></p>

</details>

#### 틀린 곳 찾기

> **[예측] 「여기부터 AI 코드」 줄 아래 코드는 오류 없이 돌아가는데 답이 틀렸습니다. 틀린 곳은 어디이고 왜일까요?**
>
> 1. `MAT[:, o - 27:o + 1]`: 입력 28일의 마지막 날이 원점 당일이다
> 2. `mlp_forecast(origin, raw28)`: 원점마다 다른 역별 평균·표준편차로 표준화한다
> 3. `qden_of(origin)`: MASE 의 분모를 원점마다 새로 구한다
>
> 답을 정한 뒤 아래를 읽어 보세요(정답 채점은 없습니다).

In [ ]:
# 미션 2 · 준비 — 읽고 실행만 합니다. 이 셀이 미션 2 에 쓸 이름을 직접 만듭니다
# [heavy] 원점 5개마다 글로벌 MLP(MSE 손실)의 가중치를 학습하는 계산이 들어 있어 CPU 로 1～2분 걸립니다
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"   # 모델과 입력을 둘 곳(GPU 또는 CPU)
torch.backends.cuda.matmul.allow_tf32 = False              # 예측도 fp32 로, GPU 가 자릿수를 줄인 계산(TF32)을 쓰지 않게

RAW_URL = ("https://github.com/studio-js/seoul-subway-ridership/raw/main/data/"
           "seoul_subway_daily_2015_2026.parquet")
CACHE = "seoul_subway_daily.parquet"          # 한 번 받아 두면 다시 받지 않습니다
if not os.path.exists(CACHE):
    pd.read_parquet(RAW_URL).to_parquet(CACHE)
#  받아지지 않으면: 위 주소의 파일을 내려받아 Colab 왼쪽 파일 목록에 올린 뒤 이 셀을 다시 실행하세요

_raw = pd.read_parquet(CACHE)                  # 「오늘의 준비」와 같은 정제와 100개 역 표를 이 셀이 다시 만듭니다
_raw["date"] = pd.to_datetime(_raw["date"])
_raw["station"] = _raw["station"].astype(str).str.replace(r"\(.*\)$", "", regex=True)
_raw["line"] = _raw["line"].astype(str).replace({"9호선2단계": "9호선(2~3단계)",
                                                "9호선2~3단계": "9호선(2~3단계)"})
_clean = _raw.groupby(["date", "line", "station"], as_index=False,
                      observed=True)[["board", "alight"]].sum()
_span = _clean.groupby(["line", "station"])["date"].agg(["min", "max", "count"])
_full = _span[(_span["min"] <= pd.Timestamp("2015-01-01")) &
              (_span["max"] >= pd.Timestamp("2026-06-30")) & (_span["count"] >= 4190)]
_top = (_clean[_clean["date"] <= pd.Timestamp("2024-12-31")].groupby(["line", "station"])["board"]
        .mean().loc[_full.index].sort_values(ascending=False).index[:100])
_wide = _clean.pivot_table(index=["line", "station"], columns="date", values="board").loc[_top]

MAT = _wide.to_numpy(dtype=float)             # 100개 역 x 4,199일 승차 행렬 (명)
DATES = pd.DatetimeIndex(_wide.columns)       # 행렬 열의 날짜
STATIONS = [f"{_a} {_b}" for _a, _b in _wide.index]   # 행렬 행의 역 이름 (0번 = 2호선 강남)
ORIGINS = pd.to_datetime(["2025-08-13", "2025-10-01", "2025-12-10",
                          "2026-02-11", "2026-06-24"])        # 원점 5개(예측 첫날)
PLAIN_ORIGINS = ORIGINS[[2, 4]]               # 공휴일 없는 두 주
HOL_ORIGINS = ORIGINS[[0, 1, 3]]              # 공휴일 있는 세 주
L, H = 28, 7                                  # 입력 28일, 채점 7일


def qden_of(origin):
    """원점 전날까지의 자료로 역마다 구한 MASE 분모(계절 나이브의 평균 절대 오차, 명)를 돌려줍니다"""
    o = DATES.get_loc(pd.Timestamp(origin))
    past = MAT[:, :o]
    return np.mean(np.abs(past[:, 7:] - past[:, :-7]), axis=1)


def score(y, pred, qden):
    """MASE(역마다 분모로 나눈 절대 오차의 평균)와 RMSE(명)를 사전으로 돌려줍니다"""
    y, pred = np.asarray(y, dtype=float), np.asarray(pred, dtype=float)
    q = np.asarray(qden, dtype=float).reshape(-1, 1)
    return dict(mase=float(np.mean(np.abs(y - pred) / q)),
                rmse=float(np.sqrt(np.mean((y - pred) ** 2))))


class _MLP(nn.Module):      # 이름 앞 밑줄: 미션 1 준비 셀의 MLP(seed 인자를 받는다)와 겹치지 않게
    """입력 28 → 64 → ReLU → 64 → ReLU → 출력 7"""
    def __init__(self, hidden=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(L, hidden), nn.ReLU(),
                                 nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, H))

    def forward(self, x):
        return self.net(x)


def _windows(Z, t_lo, t_hi):
    """첫 출력일이 t_lo ~ t_hi 인 윈도(입력 28일, 출력 7일)를 역마다 잘라 한 표로 모은다"""
    W = np.lib.stride_tricks.sliding_window_view(Z, L + H, axis=1)[:, t_lo - L:t_hi - L + 1, :]
    return (W[..., :L].reshape(-1, L).astype(np.float32),
            W[..., L:].reshape(-1, H).astype(np.float32))


def _train(model, Xtr, Ytr, Xva, Yva, batch=256, max_epochs=30, patience=5, seed=0):
    """MSE 손실과 조기 종료로 학습하고 가장 좋았던 에폭의 가중치로 되돌립니다 (CPU)"""
    np.random.seed(seed)
    torch.manual_seed(seed)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
    lossf = nn.MSELoss()
    Xtr_t, Ytr_t, Xva_t, Yva_t = map(torch.from_numpy, (Xtr, Ytr, Xva, Yva))
    gen = torch.Generator().manual_seed(seed)
    best, best_state, bad = float("inf"), None, 0
    for ep in range(1, max_epochs + 1):
        model.train()
        perm = torch.randperm(len(Xtr_t), generator=gen)
        for i in range(0, len(Xtr_t), batch):
            idx = perm[i:i + batch]
            optimizer.zero_grad()
            lossf(model(Xtr_t[idx]), Ytr_t[idx]).backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            val = lossf(model(Xva_t), Yva_t).item()
        if val < best - 1e-7:
            best, bad = val, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break
    model.load_state_dict(best_state)
    model.eval()


def _fit_weights(origin):
    """원점 전날까지의 자료만으로 글로벌 MLP 의 가중치를 학습한다 (이 셀이 대신하는 계산입니다)"""
    o = DATES.get_loc(origin)
    past = MAT[:, :o]                                   # 원점 전날까지
    cut = o - 56
    mu = past[:, :cut].mean(axis=1, keepdims=True)      # 역별 평균·표준편차: 원점 57일 전까지로 잰 값
    sd = past[:, :cut].std(axis=1, keepdims=True)
    Z = (past - mu) / sd
    Xtr, Ytr = _windows(Z, L, cut - H)                  # 학습 윈도: 목표일이 원점 57일 전까지
    Xva, Yva = _windows(Z, cut, o - H)                  # 검증 구간: 목표일이 원점 직전 56일인 윈도
    torch.manual_seed(0)
    model = _MLP()
    _train(model, Xtr, Ytr, Xva, Yva, seed=0)
    return model, mu[:, 0], sd[:, 0]


MODELS, MU, SD = {}, {}, {}                   # 원점 -> 모델(DEVICE 로 옮김) · 역별 평균 · 역별 표준편차
_t0 = time.perf_counter()
for _origin in ORIGINS:
    _m, MU[_origin], SD[_origin] = _fit_weights(_origin)
    MODELS[_origin] = _m.to(DEVICE)           # 가중치는 CPU 에서 학습하고, 예측은 DEVICE 에서 합니다
    print(f"  원점 {_origin.date()} 가중치 학습 : 원점 전날 {(_origin - pd.Timedelta(days=1)).date()} 까지의 자료"
          f" · 누적 {time.perf_counter() - _t0:.0f}초")


def mlp_forecast(origin, raw28):
    """원점 origin 의 MLP 로 역 x 28일 원자료(명)에서 역 x 7일 예측(명)을 계산해 돌려줍니다.

    표준화(MU·SD) -> DEVICE 에 있는 모델 -> .cpu() 로 가져오기 -> 명으로 되돌리기 순서입니다.
    """
    origin = pd.Timestamp(origin)
    raw28 = np.asarray(raw28, dtype=float)
    if raw28.shape[-1] != L:
        raise ValueError(f"입력은 역마다 {L}일이어야 합니다 (지금 {raw28.shape[-1]}일)")
    mu, sd = MU[origin][:, None], SD[origin][:, None]
    x = torch.as_tensor((raw28 - mu) / sd, dtype=torch.float32).to(DEVICE)   # 입력도 모델과 같은 DEVICE 로
    with torch.no_grad():
        z = MODELS[origin](x).cpu().numpy().astype(float)                   # 결과는 CPU 로 가져온다
    return z * sd + mu


# 본문 수 확인 (자료가 바뀌면 여기서 멈춥니다)
assert len(_span) == 639 and len(_full) == 521 and MAT.shape == (100, 4199) and not np.isnan(MAT).any()
_sn = {og: score(MAT[:, DATES.get_loc(og):DATES.get_loc(og) + H],
                 MAT[:, DATES.get_loc(og) - 7:DATES.get_loc(og)], qden_of(og))["mase"] for og in ORIGINS}
assert (round(np.mean([_sn[og] for og in PLAIN_ORIGINS]), 3),
        round(np.mean([_sn[og] for og in HOL_ORIGINS]), 3)) == (0.445, 2.767)   # 표 1 의 계절 나이브

# 원점 5개로 넓힌 MLP(MAE 손실) MASE: 이 교재를 만든 환경(맥 CPU)에서 원점마다 fit_global(origin, loss="mae") 로 잰 값
_MAE5 = {"2025-08-13": 1.066586, "2025-10-01": 4.322241, "2025-12-10": 0.325609,
         "2026-02-11": 2.761842, "2026-06-24": 0.392878}
_mae2 = float(np.mean([_MAE5[str(og.date())] for og in PLAIN_ORIGINS]))
_mae3 = float(np.mean([_MAE5[str(og.date())] for og in HOL_ORIGINS]))
_sn3 = float(np.mean([_sn[og] for og in HOL_ORIGINS]))
assert (round(_mae2, 3), round(_mae3, 3)) == (0.359, 2.717) and round(_sn3 - _mae3, 2) == 0.05

print(f"100개 역 표 MAT : {MAT.shape[0]}개 역 x {MAT.shape[1]:,}일 ({DATES[0].date()} ~ {DATES[-1].date()})")
print(f"글로벌 MLP 5개 : 입력 {L} -> 64 -> 64 -> 출력 {H} · MSE 손실 · 예측 DEVICE {DEVICE}")
print("쓸 수 있는 이름 : MAT · DATES · ORIGINS · PLAIN_ORIGINS · HOL_ORIGINS · L · H · qden_of(origin)"
      " · score(y, pred, qden) · mlp_forecast(origin, raw28) · DEVICE · MODELS · MU · SD")
print(f"참고 · MLP(MAE 손실) 원점 5개 (교재를 만든 환경) : 공휴일 없는 두 주 평균 {_mae2:.3f}"
      f" · 공휴일 있는 세 주 평균 {_mae3:.3f} (계절 나이브 0.445 · {_sn3:.3f})")

In [ ]:
# 미션 2 — 프롬프트 (함정)   # ai: prompt
# 이 미션은 함정입니다. 아래는 전임 담당자가 남긴 코드라 돌아가지만 답이 틀렸습니다.
# 어디를 어떻게 고칠지 PROMPT 에 쓰면 튜터가 AI 코드 줄 아래를 다시 씁니다.
# 고칠 지시는 AI 튜터 대화창이 아니라 아래 따옴표 안에 적습니다. 대화창에만 쓰면 진단은 나오지만 완성본 풀이가 열리지 않습니다
PROMPT = """

"""
# --- 여기부터 AI 코드 ---
# ↓ 전임 담당자의 코드. 그대로 실행하면 채점이 ❌ 를 냅니다
mase_by, gap_by = {}, {}
for origin in ORIGINS:
    o = DATES.get_loc(origin)
    raw28 = MAT[:, o - 27:o + 1]              # 가장 최근 28일을 넣어야 정확하다: 원점 당일까지
    pred = mlp_forecast(origin, raw28)
    mase_by[origin] = score(MAT[:, o:o + H], pred, qden_of(origin))["mase"]
    first, last = DATES[o - 27], DATES[o]
    gap_by[origin] = (origin - last).days     # 입력 끝날에서 원점까지 며칠인지
    print(f"원점 {origin.date()} | 입력 윈도 {first.date()} ~ {last.date()}"
          f" | 끝날에서 원점까지 {gap_by[origin]}일 | MASE {mase_by[origin]:.3f}")
print(f"원점 전날에서 끝나는 입력 윈도 수 = {sum(g == 1 for g in gap_by.values())}")
print(f"공휴일 없는 두 주 평균 MASE = {np.mean([mase_by[o] for o in PLAIN_ORIGINS]):.3f}")
print(f"  글로벌 MLP 평균 MASE {np.mean(list(mase_by.values())):.3f}: 계절 나이브 1.838 보다 크게 빗나갑니다")
print("판단 : 신경망은 버립니다")

fig, ax = plt.subplots(figsize=(FIG_W * 0.8, 3.6))
ax.bar([str(o.date()) for o in ORIGINS], list(mase_by.values()), color=COL["연회색"], width=0.6)
ax.set_xlabel("원점")
ax.set_ylabel("MASE")
plt.show()

## 오늘의 결론

<div class="lms-summary">

**정리**

- 손실만 바꾼 두 모델: 원점 2025-12-10(공휴일 없는 주)에서 MLP(MAE 손실)은 MLP(MSE 손실)보다 약 0.2 낮습니다. 계절 나이브 0.419 보다도 낮습니다. 원점 2025-10-01(공휴일 있는 주)에서는 MLP(MAE 손실)이 약 0.28 높습니다
- 원점 5개 두 묶음: 「미션 2 준비 셀」 참고 줄의 MLP(MAE 손실)은 공휴일 없는 두 주 0.359 로 계절 나이브 0.445 보다 낮습니다. 공휴일 있는 세 주 평균 2.717 은 계절 나이브 2.767 과 0.05 차이라 낫다고 보기 어렵습니다
- 입력 윈도 누수: 원점 전날에서 끝나게 고치면 MLP(MSE 손실)의 공휴일 없는 두 주 평균은 약 0.739 입니다. 계절 나이브 0.445 보다 큽니다. 원점 당일이 입력에 포함된 전임자 코드는 약 2.13 이라, 누수는 점수가 아니라 입력 윈도의 날짜로 확인합니다
- 할 일: `fit_global(origin, loss="mae")` 로 학습하고, 원점마다 입력 윈도 끝날이 원점 전날인지 출력해 둡니다. 평가표에는 모델마다 평균과 두 묶음 평균을 함께 적고, 평일 공휴일이 있는 주는 담당자가 인력을 따로 정합니다

공휴일 없는 주의 역별 인력 예측 후보는 손실을 MAE 로 둔 글로벌 MLP 입니다. 평일 공휴일이 있는 주는 계절 나이브와 비슷해 이 모델로 정하지 않습니다.

</div>

> **[과제] 실습 제출 — 손실을 바꾼 두 모델과 입력 윈도 누수 수정**
>
> 미션 1·2 의 「PROMPT 셀」이 채점을 통과한 상태로 노트북을 제출합니다. PROMPT 는 내가 쓴 그대로 둡니다.
>
> 마감: 2026-10-16T23:59+09:00
>
> **평가 기준**
>
> | 기준 | 이렇게 하면 충족 | 배점 |
> | --- | --- | --- |
> | 미션 1 | 채점을 통과했고 PROMPT 에 만들 것이 내 말로 적혀 있다 | 50 |
> | 미션 2 | 채점을 통과했고 PROMPT 에 고칠 것이 내 말로 적혀 있다 | 50 |
>
> 제출은 LMS 레슨 화면의 과제 카드에서 GitHub 링크로 합니다(이 파일에 적으면 제출되지 않습니다).

<div style="background:#F2F2F2;color:#000000;padding:28px;border-radius:12px;border:1px solid #E6E6E6;margin-top:16px;border-left:4px solid #051C2C">
<h2 style="color:#000000;margin:0 0 8px 0">수고했습니다 — 딥러닝 시계열 첫날 완료</h2>
<div style="color:#656565">손실만 바꾼 글로벌 MLP 두 개를 공휴일 없는 주와 공휴일 있는 주에서 계절 나이브와 비교했습니다. 원점 당일이 입력에 포함된 누수는 입력 윈도의 날짜로 찾아 고쳤습니다.</div>
</div>